# 01 · Datos, validación y experimentos de recomendación (v2)

**Grupo 5 · JTH · versión ampliada del notebook de Andrés (5 de noviembre de 2026)**

Este notebook contiene **todo** el trabajo de `01_datos_y_validacion.ipynb`, reproducido, más los experimentos que faltaban para alinearlo con la propuesta. La versión original se mantiene sin cambios en el repositorio. `scripts/check_v2_reproduces_v1.py` compara celda por celda las salidas de ambas.

**Qué cambió respecto de la versión original.**

- Los mecanismos compartidos pasaron a `src/jth/`: consultas y elegibilidad (`context`), rankings y métricas (`ranking`), actividad y filtros (`activity`), fusiones (`fusion`), modelo de actividad (`activity_model`), CF sin catálogo transductivo (`clean_cf`) e intervalos (`uncertainty`). Las corridas oficiales pasaron a `scripts/run_references.py`.
- La exploración ahora va antes de los modelos. La tabla siguiente indica dónde quedó cada sección.
- Se agregaron comparaciones junto a las originales, que se conservan. Cada adición indica qué pregunta responde.

| Secciones aquí | Contenido | Secciones en la versión original |
|---|---|---|
| 1–9 | Preparación, partición y validación | 1–9 |
| 10–22 | Exploración de atributos, calidad, historial y actividad | 16–28 |
| 23–27 | Recencia, contenido, LR y CF, actividad, fusión | 10–14 |
| 28–32 | **Nuevo:** referencias que faltaban, filtros en cada modelo, P(actividad), mezclas A, B y C, orden del filtro | |
| 33 | Comparación de la etapa e incertidumbre | 15 |
| 34–37 | **Nuevo:** contribución de componentes, selección en dos tramos, congelamiento y test | |
| 38 | Síntesis y próximos pasos | 29 |

**Cómo abrirlo:** en VS Code, selecciona como kernel el Python de `.venv` del repositorio, creado con `uv sync`. El notebook funciona abriéndolo desde la raíz del repositorio o desde `scripts`. Las corridas largas quedan en una caché verificada dentro de `data/cache/`, que git ignora.


## 1. ¿Qué estamos construyendo?

Queremos recomendar ofertas a un candidato en una fecha concreta. Cada modelo asignará puntuaciones a las ofertas y devolverá una lista ordenada. Evaluaremos dónde queda la oferta asociada al proceso observado.

Para replicar JTH, un positivo es un **proceso candidato–oferta registrado**, cualquiera sea su etapa final. No significa necesariamente contratación ni una preferencia explícita del candidato.

Antes de modelar necesitamos saber qué datos corresponden a entrenamiento, qué bloque servirá para elegir parámetros y cuál se conserva para la evaluación final. Una interacción es una fila de proceso; una consulta es una combinación candidato–día. Estos conceptos no siempre tienen el mismo conteo.


## 2. Preparar el entorno del notebook

Localizamos el repositorio a partir de la carpeta de ejecución y cargamos las librerías ya instaladas. La tabla siguiente muestra qué Python y qué versiones estamos utilizando. No se instalan paquetes desde el notebook.


In [ ]:
from pathlib import Path
import sys
import subprocess
import tempfile
import hashlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline

inicio = Path.cwd().resolve()
RAIZ = next(
    (ruta for ruta in [inicio, *inicio.parents]
     if (ruta / 'pyproject.toml').is_file() and (ruta / 'src' / 'jth').is_dir()),
    None,
)
if RAIZ is None:
    raise RuntimeError('Abre el notebook desde el repositorio jth-recsys o desde scripts.')

RAW = RAIZ / 'data' / 'raw'
DIVISOR_OFICIAL = RAIZ / 'external' / 'JTH' / 'code' / 'utils' / 'binary_splitter.py'
assert DIVISOR_OFICIAL.is_file(), 'Falta inicializar el submódulo oficial.'

display(pd.DataFrame({
    'Elemento': ['Raíz del repositorio', 'Python utilizado', 'Versión de Python', 'pandas', 'numpy'],
    'Valor': [RAIZ.name, Path(sys.executable).name, sys.version.split()[0], pd.__version__, np.__version__],
}))


## 3. Leer las tres tablas y observar su organización

`candidates.csv` describe candidatos, `jobs.csv` describe ofertas y `history.csv` contiene procesos entre ambos. Leemos los IDs como texto para no cambiar su representación. Primero mostramos tamaños y nombres de columnas; no hace falta mostrar perfiles individuales para entender la estructura.


In [ ]:
candidatos = pd.read_csv(RAW / 'candidates.csv', dtype=str)
ofertas = pd.read_csv(RAW / 'jobs.csv', dtype=str)
historial = pd.read_csv(RAW / 'history.csv', dtype=str)

tablas = {'candidates.csv': candidatos, 'jobs.csv': ofertas, 'history.csv': historial}
display(pd.DataFrame([
    {'Archivo': nombre, 'Filas': len(tabla), 'Columnas': len(tabla.columns)}
    for nombre, tabla in tablas.items()
]))


**Lectura del resultado.** En esta versión hay 37.554 perfiles de candidatos, 6.011 ofertas y 42.288 procesos. No todas las entidades del catálogo tienen un proceso registrado. Una tabla de perfiles y una tabla de interacciones responden preguntas distintas.

Las columnas de `history.csv` permiten ver qué conecta cada proceso y cuáles son las fechas disponibles. El inventario de nombres siguiente no muestra datos personales.


In [ ]:
display(pd.DataFrame({'Columna de history.csv': historial.columns}))

display(pd.DataFrame([
    {'Entidad': nombre, 'IDs únicos': tabla[columna].nunique(),
     'IDs duplicados': int(tabla[columna].duplicated().sum()),
     'IDs faltantes': int(tabla[columna].isna().sum())}
    for nombre, tabla, columna in [
        ('Candidato', candidatos, 'candidate_id'),
        ('Oferta', ofertas, 'job_id'),
    ]
]))


## 4. ¿Cómo se fecha una interacción?

El divisor oficial usa la fecha mínima entre las columnas que contienen `_date`, excluye las que contienen `KO` y excluye `job_offer_opening_date`. Es la primera etapa registrada del proceso, no su fecha de contratación.

En esta celda mostramos las columnas utilizadas y comprobamos que se puede asignar una fecha a cada proceso. No usamos la etapa máxima alcanzada como característica para predecir el comienzo del proceso.


In [ ]:
columnas_fecha = [
    nombre for nombre in historial.columns
    if '_date' in nombre and 'KO' not in nombre and nombre != 'job_offer_opening_date'
]
fechas = historial[columnas_fecha].apply(pd.to_datetime)
primera_fecha = fechas.min(axis=1)

assert primera_fecha.notna().all(), 'Hay procesos sin fecha utilizable.'
assert historial[['candidate_id', 'job_id']].notna().all().all()

display(pd.DataFrame({'Fechas consideradas por el divisor oficial': columnas_fecha}))
display(pd.DataFrame({
    'Medida': ['Procesos con fecha', 'Primera fecha', 'Última fecha'],
    'Valor': [int(primera_fecha.notna().sum()),
              primera_fecha.min().date(), primera_fecha.max().date()],
}))


## 5. Reproducir la partición oficial 80/20

Para mantener la comparación con el paper, ejecutamos el **divisor oficial sin modificarlo**. Ordena los procesos por su primera fecha y asigna el 80 % inicial al entrenamiento y el resto al test.

El programa oficial escribe CSV. Para ver todo aquí sin dejar archivos auxiliares en el repositorio, lo ejecutamos dentro de una carpeta temporal, cargamos sus resultados en memoria y eliminamos esa carpeta automáticamente al salir del bloque `with`.

`train_oficial` y `test_oficial` serán DataFrames que permanecerán disponibles para las siguientes celdas.


In [ ]:
hash_divisor = hashlib.sha256(DIVISOR_OFICIAL.read_bytes()).hexdigest()

with tempfile.TemporaryDirectory(prefix='jth_notebook_split_') as carpeta_temporal:
    ejecucion = subprocess.run(
        [sys.executable, str(DIVISOR_OFICIAL), str(RAW / 'history.csv'), carpeta_temporal],
        check=True, capture_output=True, text=True,
    )
    train_oficial = pd.read_csv(
        Path(carpeta_temporal) / 'train.csv', dtype=str, parse_dates=['timestamp']
    )
    test_oficial = pd.read_csv(
        Path(carpeta_temporal) / 'test.csv', dtype=str, parse_dates=['timestamp']
    )

def resumir(nombre, datos):
    dias = datos['timestamp'].dt.normalize()
    return {
        'Bloque': nombre,
        'Procesos': len(datos),
        'Consultas candidato-día': datos.assign(dia=dias).groupby(['candidate_id', 'dia']).ngroups,
        'Desde': dias.min().date(),
        'Hasta': dias.max().date(),
    }

display(pd.DataFrame([
    resumir('Entrenamiento oficial', train_oficial),
    resumir('Test oficial', test_oficial),
]))
print('SHA256 del divisor oficial:', hash_divisor)


**Resultado.** El entrenamiento oficial contiene 33.830 procesos y el test contiene 8.458. Son las cantidades que aparecían en la propuesta. El entrenamiento oficial completo se utilizará para ajustar el modelo final una vez elegidos sus parámetros.

No elegimos ventanas o pesos mirando los resultados del test. La siguiente separación de validación se hace exclusivamente dentro del entrenamiento oficial.


## 6. ¿Por qué 8.458 procesos no son 8.458 consultas?

Una misma persona puede tener dos procesos distintos el mismo día. En ese caso hay dos filas de interacción, pero una sola clave candidato–día.

El ejemplo siguiente es **inventado** y sirve para visualizarlo. Después contamos las claves en el test real. El evaluador oficial conserva un solo positivo cuando se repite una clave; el evaluador local admite varios. Por eso debemos mantener visibles las diferencias de evaluación.


In [ ]:
ejemplo = pd.DataFrame({
    'candidate_id': ['candidato_A', 'candidato_A', 'candidato_B'],
    'job_id': ['oferta_1', 'oferta_2', 'oferta_3'],
    'dia': ['2020-01-10', '2020-01-10', '2020-01-10'],
})
display(ejemplo)
print('Ejemplo: procesos =', len(ejemplo),
      '| consultas =', ejemplo.groupby(['candidate_id', 'dia']).ngroups)

filas_por_consulta_test = test_oficial.assign(
    dia=test_oficial['timestamp'].dt.normalize()
).groupby(['candidate_id', 'dia']).size()

display(pd.DataFrame({
    'Medida en test': [
        'Procesos', 'Consultas distintas',
        'Filas adicionales respecto de las claves', 'Consultas con más de un proceso',
    ],
    'Cantidad': [len(test_oficial), len(filas_por_consulta_test),
                 len(test_oficial) - len(filas_por_consulta_test),
                 int((filas_por_consulta_test > 1).sum())],
}))


**Interpretación.** Hay 8.341 consultas de test. La diferencia de 117 corresponde a filas adicionales respecto de las claves distintas; no debemos confundirla con el número de consultas repetidas. La tabla anterior calcula ambas cantidades por separado.

La misma distinción explicará enseguida por qué la validación tiene 4.231 procesos y 4.208 consultas.


## 7. Decisión de trabajo: reservar validación dentro del entrenamiento

Reservamos aproximadamente el **último 12,5 % del entrenamiento oficial**, moviendo el corte a una frontera de día. Aquí reconstruimos esa decisión de manera visible y revisable.

- **Entrenamiento reducido:** sirve para aprender modelos y transformaciones como vocabulario e IDF.
- **Validación:** sirve para comparar variantes y elegir parámetros.
- **Test oficial:** se conserva para evaluar la configuración final.

La posición del corte se calcula sobre las filas del entrenamiento, pero todas las filas del día elegido pasan a validación. Por eso su tamaño puede diferir ligeramente de un 12,5 % exacto. Esta celda no escribe ningún archivo.


In [ ]:
FRACCION_VALIDACION = 0.125
assert 0 < FRACCION_VALIDACION < 1

dias_train = train_oficial['timestamp'].dt.normalize()
posicion_corte = int(len(train_oficial) * (1 - FRACCION_VALIDACION))
dia_corte = dias_train.sort_values(kind='mergesort').iloc[posicion_corte]

es_entrenamiento = dias_train < dia_corte
train_reducido = train_oficial.loc[es_entrenamiento].copy()
validacion = train_oficial.loc[~es_entrenamiento].copy()

print('Día de corte:', dia_corte.date())
print(f'Porcentaje efectivo de validación dentro del train oficial: {len(validacion) / len(train_oficial):.2%}')
display(pd.DataFrame([
    resumir('Entrenamiento reducido', train_reducido),
    resumir('Validación', validacion),
    resumir('Test oficial', test_oficial),
]))


**De aquí salen las 4.208 consultas.** La celda anterior toma las 4.231 filas de validación y cuenta sus combinaciones distintas de candidato y día. No es un número impuesto manualmente.

Con los datos y la fracción actuales, el corte es el **21 de julio de 2023**. Quedan 29.599 procesos para entrenamiento reducido y 4.231 para validación. En el primer entrenamiento no se usan estos últimos procesos; una LR entrenada con todo el entrenamiento oficial no serviría para evaluar honestamente esta validación.


## 8. Verificar la separación y conservar todas las filas

Comprobamos que los dos bloques reconstruyen el entrenamiento oficial, que no comparten días y que sus períodos están ordenados. Si una condición falla, detenemos el notebook mediante una aserción.

También revisamos una particularidad del benchmark: el corte oficial 80/20 puede compartir un día entre train y test porque se hace por filas. Lo registramos sin cambiar silenciosamente el protocolo del paper.


In [ ]:
assert not train_reducido.empty and not validacion.empty
reconstruido = pd.concat([train_reducido, validacion]).sort_index()
pd.testing.assert_frame_equal(reconstruido, train_oficial)

dias_reducido = set(train_reducido['timestamp'].dt.normalize())
dias_validacion = set(validacion['timestamp'].dt.normalize())
assert dias_reducido.isdisjoint(dias_validacion)
assert max(dias_reducido) < min(dias_validacion)

dias_compartidos_oficiales = sorted(
    set(train_oficial['timestamp'].dt.normalize())
    & set(test_oficial['timestamp'].dt.normalize())
)

display(pd.DataFrame({
    'Comprobación': ['Filas del train conservadas', 'Días comunes entre train reducido y validación',
                     'Último día de train reducido anterior a validación'],
    'Resultado': [True, len(dias_reducido & dias_validacion), True],
}))
print('Días compartidos por train/test oficiales:', [str(d.date()) for d in dias_compartidos_oficiales])


**Implicación.** La validación acordada tiene una frontera diaria estricta. La partición oficial, en cambio, comparte el 23 de marzo de 2024 entre entrenamiento y test. Conservamos esta particularidad para replicar el benchmark y no afirmamos que esa frontera oficial sea estricta por día.

Durante la evaluación diaria, un modelo debe puntuar todas las consultas del día antes de incorporar sus interacciones. La actualización diaria del historial no autoriza a elegir parámetros con resultados del test.


## 9. Visualizar la línea de tiempo

El gráfico siguiente muestra el volumen mensual de procesos y colorea los períodos de entrenamiento reducido, validación y test. Se dibuja dentro del notebook, sin generar un PNG externo. Los meses extremos pueden contener datos parciales.


In [ ]:
todos_los_procesos = pd.concat([train_oficial, test_oficial], ignore_index=True)
volumen_mensual = todos_los_procesos.set_index('timestamp').resample('MS').size()
inicio_test = test_oficial['timestamp'].min()
fin_datos = test_oficial['timestamp'].max() + pd.Timedelta(days=1)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.axvspan(train_oficial['timestamp'].min(), dia_corte,
           color='#2563eb', alpha=0.10, label='Entrenamiento reducido')
ax.axvspan(dia_corte, inicio_test, color='#f59e0b', alpha=0.18, label='Validación')
ax.axvspan(inicio_test, fin_datos, color='#16a34a', alpha=0.12, label='Test oficial')
ax.plot(volumen_mensual.index, volumen_mensual.values, color='#334155', linewidth=1.5)
ax.axvline(dia_corte, color='#b45309', linestyle='--', linewidth=1)
ax.axvline(inicio_test, color='#15803d', linestyle='--', linewidth=1)
ax.set(title='Línea de tiempo de los datos y los bloques de trabajo',
       xlabel='Fecha', ylabel='Procesos por mes')
ax.grid(alpha=0.2)
ax.legend(loc='upper left', fontsize=9)
fig.tight_layout()
plt.show()


**Lectura del gráfico.** Los bloques están ordenados en el tiempo. El volumen cambia entre períodos, de modo que una división aleatoria de filas no representaría la misma tarea de recomendar hacia el futuro. El gráfico muestra actividad observada, no la duración real de apertura de las ofertas.


In [ ]:
import json
import random
import time
import importlib.util
from IPython.display import Markdown

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 10. Segunda iteración: preguntas que orientan la exploración

Profundizaremos en **composición, calidad, contenido y tiempo**, manteniendo la secuencia de explicación, código e interpretación. Las figuras se calculan con pandas y se dibujan con matplotlib dentro del notebook.

| Pregunta | Datos utilizados | Relación con el recomendador |
|---|---|---|
| ¿Qué atributos existen y cuánto falta? | Esquema y cobertura de los CSV completos | Diseñar el preprocesamiento, sin aprender parámetros con todo el catálogo |
| ¿Hay duplicados o inconsistencias? | Auditoría de los CSV completos | Entender la unidad de interacción antes de eliminar o transformar filas |
| ¿Cuánto historial hay por entidad? | Entrenamiento reducido | Evaluar la viabilidad de CF y del perfil de oferta |
| ¿Qué contenido comparten candidatos y ofertas? | Perfiles de entidades con procesos en entrenamiento reducido | Explorar habilidades y formatos para el baseline de contenido |
| ¿Cómo cambian actividad y cold-start? | Entrenamiento reducido y recorrido diario de validación | Elegir hipótesis temporales sin seleccionar parámetros con test |
| ¿Qué cobertura tendría una ventana de actividad? | Validación, usando únicamente su pasado en cada día | Medir pérdidas potenciales antes de implementar un filtro |

Los análisis de perfiles completos son **descriptivos de la instantánea publicada**. No autorizan a ajustar IDF, imputaciones o modelos con perfiles futuros. No mostramos filas de personas reales ni usamos atributos demográficos como features. El test conserva su papel en el protocolo descrito arriba; esta iteración no lo usa para escoger variantes.


## 11. Composición de columnas e inventario de atributos

Todos los CSV se leyeron como texto para preservar los IDs. Ese dtype de lectura no significa que todas las variables sean semánticamente iguales. Distinguimos identificadores, fechas, atributos de reclutador, atributos inferidos por LLM y campos de proceso.

El inventario desplegable permite consultar **todas las columnas**, su disponibilidad y cardinalidad. Una respuesta `False` o un valor cero cuentan como datos presentes; no se consideran ausencias. La disponibilidad no mide exactitud ni relevancia predictiva.


In [ ]:
from IPython.display import HTML, Markdown
from matplotlib.ticker import PercentFormatter

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

SENSIBLES = {'llm_sex', 'llm_nationality', 'llm_age_bucket'}

def presente(serie):
    return serie.notna() & serie.fillna('').astype(str).str.strip().ne('')

def grupo_campo(nombre, archivo):
    if nombre.endswith('_id'):
        return 'Identificador'
    if nombre == 'create_date' or nombre.endswith('_date'):
        return 'Fecha'
    if nombre.startswith('llm_'):
        return 'Atributo LLM'
    return 'Campo de proceso' if archivo == 'history.csv' else 'Atributo de reclutador/origen'

inventario = pd.DataFrame([
    {'Archivo': archivo, 'Columna': columna, 'Grupo': grupo_campo(columna, archivo),
     'Disponibilidad (%)': 100 * presente(tabla[columna]).mean(),
     'Valores distintos presentes': tabla.loc[presente(tabla[columna]), columna].nunique(),
     'Excluir como feature demográfica': columna in SENSIBLES}
    for archivo, tabla in tablas.items() for columna in tabla.columns
])
composicion = pd.crosstab(inventario['Archivo'], inventario['Grupo'])
display(composicion)
ax = composicion.plot.bar(stacked=True, figsize=(10, 3.8), colormap='tab20')
ax.set(xlabel='', ylabel='Número de columnas', title='Composición del esquema de cada archivo')
ax.tick_params(axis='x', rotation=0)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()
display(HTML('<details><summary>Desplegar inventario completo de columnas</summary>'
             + inventario.round(2).to_html(index=False) + '</details>'))


**Implicación para el proyecto.** La abundancia de columnas LLM no garantiza que aporten señal independiente. Varias describen el mismo concepto que un atributo humano y pueden discrepar. Los IDs sirven para unir tablas; no son números ordinales. Los resultados finales del proceso tampoco son atributos conocidos al recomendar su primera etapa.


## 12. Datos faltantes y complementariedad entre fuentes

Mostramos los atributos con más ausencias y comparamos conceptos presentes tanto en campos humanos como en campos LLM. La tabla distingue cuatro casos: ambas fuentes, solo humano, solo LLM y ninguna.

El porcentaje se calcula sobre **todos los perfiles de cada archivo**. Esto describe cobertura de la instantánea, no disponibilidad histórica en una fecha ni precisión de las inferencias.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, archivo in zip(axes, ['candidates.csv', 'jobs.csv']):
    subconjunto = inventario[
        (inventario['Archivo'] == archivo)
        & inventario['Grupo'].isin(['Atributo LLM', 'Atributo de reclutador/origen'])
        & ~inventario['Excluir como feature demográfica']
    ].copy()
    faltantes = (100 - subconjunto.set_index('Columna')['Disponibilidad (%)']).nlargest(10)
    faltantes.sort_values().plot.barh(ax=ax, color='#527aa5')
    ax.set(title=f'Ausencias en {archivo}', xlabel='Perfiles sin valor (%)', ylabel='')
    ax.set_xlim(0, 100)
    ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
plt.show()

pares_atributos = [
    ('Habilidades', 'skills', 'llm_hard_skills', 'skills', 'llm_hard_skills'),
    ('Área de experiencia', 'expertise_area', 'llm_expertise_area', 'expertise_area', 'llm_expertise_area'),
    ('Categoría', 'job_category', 'llm_job_category', 'job_category', 'llm_job_category'),
    ('Experiencia', 'years_experience', 'llm_years_of_work_experience',
     'years_experience', 'llm_required_years_of_work_experience'),
    ('Salario', 'actual_salary', 'llm_estimated_salary_eur_per_year',
     'salary', 'llm_salary_estimation_eur_per_year'),
]
filas_cobertura = []
for concepto, ch, cl, jh, jl in pares_atributos:
    for entidad, tabla, humano, llm in [
        ('Candidatos', candidatos, ch, cl), ('Ofertas', ofertas, jh, jl)
    ]:
        h, l = presente(tabla[humano]), presente(tabla[llm])
        filas_cobertura.append({
            'Entidad': entidad, 'Concepto': concepto,
            'Ambas (%)': 100 * (h & l).mean(), 'Solo humano (%)': 100 * (h & ~l).mean(),
            'Solo LLM (%)': 100 * (~h & l).mean(), 'Ninguna (%)': 100 * (~h & ~l).mean(),
        })
cobertura_atributos = pd.DataFrame(filas_cobertura)
display(cobertura_atributos.round(1))
assert np.allclose(cobertura_atributos.filter(like='(%)').sum(axis=1), 100)


**Interpretación.** Un campo inferido puede ampliar cobertura donde falta el humano, pero no es una observación adicional del resultado de contratación ni un reemplazo automáticamente fiable. En el baseline de contenido podremos comparar fuentes y tratar explícitamente perfiles sin tokens. No imputaremos salarios o experiencia usando todo el dataset para entrenar.


## 13. Auditoría de integridad, repeticiones y fechas

Comprobamos identificadores, enlaces entre tablas y fechas. Una fila repetida exactamente, un `application_id` repetido y un par candidato–oferta repetido son situaciones diferentes. El mismo par puede identificar varios procesos; no lo eliminamos automáticamente.

También contamos procesos anteriores a la creación declarada de sus entidades y diferencias en las fechas de etapas. Las fechas fueron anonimizadas; una anomalía requiere investigar su semántica, no corregirla por intuición. En todos los casos reportamos el denominador.


In [ ]:
creacion_candidatos = pd.to_datetime(candidatos.set_index('candidate_id')['create_date'], errors='coerce')
creacion_ofertas = pd.to_datetime(ofertas.set_index('job_id')['create_date'], errors='coerce')
fecha_candidato_proceso = historial['candidate_id'].map(creacion_candidatos)
fecha_oferta_proceso = historial['job_id'].map(creacion_ofertas)

etapas_ordenadas = [
    'spontaneous_application_date', 'shortlist_date', 'qualification_date',
    'resume_sent_to_company_date', '1st_interview_date', '2nd_interview_date',
    '3rd_interview_date', '4th_interview_date', 'job_offer_proposed_date', 'job_offer_accepted_date',
]
fechas_etapas = fechas[etapas_ordenadas]
retroceso_etapas = fechas_etapas.ffill(axis=1).diff(axis=1).lt(pd.Timedelta(0)).any(axis=1)
fechas_invalidas = sum(
    (presente(tabla['create_date']) & pd.to_datetime(tabla['create_date'], errors='coerce').isna()).sum()
    for tabla in [candidatos, ofertas]
)

filas_auditoria = [
    ('IDs candidatos duplicados', candidatos['candidate_id'].duplicated().sum(), len(candidatos)),
    ('IDs ofertas duplicados', ofertas['job_id'].duplicated().sum(), len(ofertas)),
    ('IDs de proceso duplicados', historial['application_id'].duplicated().sum(), len(historial)),
    ('Filas de historial exactamente duplicadas', historial.duplicated().sum(), len(historial)),
    ('Filas adicionales de pares candidato-oferta repetidos',
     historial.duplicated(['candidate_id', 'job_id']).sum(), len(historial)),
    ('Procesos con candidato sin perfil', (~historial['candidate_id'].isin(candidatos['candidate_id'])).sum(), len(historial)),
    ('Procesos con oferta sin perfil', (~historial['job_id'].isin(ofertas['job_id'])).sum(), len(historial)),
    ('Candidatos sin create_date', creacion_candidatos.isna().sum(), len(candidatos)),
    ('Ofertas sin create_date', creacion_ofertas.isna().sum(), len(ofertas)),
    ('Fechas de creación presentes no parseables', fechas_invalidas, len(candidatos) + len(ofertas)),
    ('Procesos anteriores a creación del candidato',
     (primera_fecha < fecha_candidato_proceso).sum(), fecha_candidato_proceso.notna().sum()),
    ('Procesos anteriores a creación de la oferta',
     (primera_fecha < fecha_oferta_proceso).sum(), fecha_oferta_proceso.notna().sum()),
    ('Procesos con retroceso entre etapas observadas', retroceso_etapas.sum(), len(historial)),
    ('Fin de proceso anterior a última etapa observada',
     (fechas['end_of_process_date'] < fechas_etapas.max(axis=1)).sum(),
     fechas['end_of_process_date'].notna().sum()),
]
auditoria = pd.DataFrame(filas_auditoria, columns=['Control', 'Casos', 'Denominador'])
auditoria['Porcentaje'] = 100 * auditoria['Casos'] / auditoria['Denominador'].replace(0, np.nan)
display(auditoria.round(3))

por_par = historial.groupby(['candidate_id', 'job_id']).size()
display(pd.DataFrame({
    'Medida': ['Pares distintos', 'Pares con varios procesos', 'Máximo de procesos por par'],
    'Cantidad': [len(por_par), int((por_par > 1).sum()), int(por_par.max())],
}))


**Consecuencia para el modelado.** Conservamos la unidad de proceso del benchmark. Un CF binario puede colapsar pares como parte de su representación, mientras un baseline de actividad cuenta eventos; esas elecciones deben documentarse porque no significan lo mismo. Las fechas de creación faltantes requieren una política explícita de elegibilidad. Este análisis no modifica datos ni elimina consultas problemáticas.


## 14. Participación, dispersión y colas largas

Primero contrastamos el catálogo completo con las entidades que tienen al menos un proceso en todo el registro. Es una descripción retrospectiva del archivo, no el catálogo disponible en una fecha.

Para estudiar la señal que un modelo inicial puede aprender, calculamos después las distribuciones usando **solo entrenamiento reducido**. La densidad usa pares distintos; informamos por separado el número de procesos para no confundir repeticiones con celdas diferentes de una matriz.


In [ ]:
participacion = pd.DataFrame([
    {'Entidad': 'Candidatos', 'Con procesos': historial['candidate_id'].nunique(),
     'Sin procesos': len(candidatos) - historial['candidate_id'].nunique()},
    {'Entidad': 'Ofertas', 'Con procesos': historial['job_id'].nunique(),
     'Sin procesos': len(ofertas) - historial['job_id'].nunique()},
]).set_index('Entidad')
display(participacion)
ax = participacion.div(participacion.sum(axis=1), axis=0).mul(100).plot.barh(
    stacked=True, figsize=(8, 2.5), color=['#2563eb', '#d1d5db']
)
ax.set(xlabel='Porcentaje del archivo de perfiles', ylabel='',
       title='Entidades con y sin procesos en todo el registro')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

def estructura(nombre, datos):
    nc, nj = datos['candidate_id'].nunique(), datos['job_id'].nunique()
    pares = len(datos.drop_duplicates(['candidate_id', 'job_id']))
    return {'Bloque': nombre, 'Procesos': len(datos), 'Pares distintos': pares,
            'Candidatos con historial': nc, 'Ofertas con historial': nj,
            'Densidad sobre entidades con historial (%)': 100 * pares / (nc * nj)}

display(pd.DataFrame([
    estructura('Registro completo · descriptivo', todos_los_procesos),
    estructura('Entrenamiento reducido', train_reducido),
    estructura('Validación · solo su composición', validacion),
]).round(4))

grados_candidatos = train_reducido.groupby('candidate_id').size()
grados_ofertas = train_reducido.groupby('job_id').size()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
resumen_grados = []
for ax, nombre, grados in [
    (axes[0], 'Candidatos', grados_candidatos), (axes[1], 'Ofertas', grados_ofertas)
]:
    grupos = pd.cut(grados, [0, 1, 2, 5, 10, np.inf], labels=['1', '2', '3–5', '6–10', '11+'])
    (grupos.value_counts(sort=False) / len(grados) * 100).plot.bar(ax=ax, color='#527aa5', rot=0)
    ax.set(title=nombre, xlabel='Procesos por entidad con historial', ylabel='Entidades (%)')
    ax.grid(axis='y', alpha=0.2)
    ntop = max(1, int(np.ceil(len(grados) * 0.2)))
    resumen_grados.append({
        'Entidad': nombre, 'Mínimo': grados.min(), 'Mediana': grados.median(),
        'Promedio': grados.mean(), 'P90': grados.quantile(.9), 'Máximo': grados.max(),
        'Con un solo proceso (%)': 100 * grados.eq(1).mean(),
        'Procesos concentrados en el 20% más activo (%)': 100 * grados.nlargest(ntop).sum() / grados.sum(),
    })
fig.suptitle('Distribución del historial en entrenamiento reducido', y=1.02)
fig.tight_layout()
plt.show()
resumen_grados = pd.DataFrame(resumen_grados)
display(resumen_grados.round(2))


**Implicación.** Un candidato con un único proceso ofrece poca señal colaborativa. Las ofertas pueden acumular más ejemplos y permitir un perfil basado en candidatos previos. La cola larga obliga a evaluar por cantidad de historial; una media global puede ocultar muchos casos casi sin datos. Una entidad sin procesos en el archivo no es un ejemplo negativo observado ni necesariamente una oferta cerrada.


## 15. Formatos de experiencia, contratos y salario

Exploramos perfiles de entidades **con procesos en entrenamiento reducido**. Este subconjunto describe entidades observadas por ese historial; no representa todos los perfiles del mercado ni autoriza suponer que sus atributos tengan versiones históricas.

La experiencia humana usa categorías como `0-2 years`, y la experiencia LLM de candidatos usa intervalos como `5-10`; la experiencia requerida LLM de ofertas sí puede ser numérica. La conversión automática de todos esos campos a número perdería información. No asignamos puntos medios a los intervalos en esta exploración.


In [ ]:
candidatos_train = candidatos[candidatos['candidate_id'].isin(train_reducido['candidate_id'])].copy()
ofertas_train = ofertas[ofertas['job_id'].isin(train_reducido['job_id'])].copy()

filas_formato = []
for entidad, tabla, campos in [
    ('Candidatos', candidatos_train, ['years_experience', 'llm_years_of_work_experience', 'actual_salary']),
    ('Ofertas', ofertas_train, ['years_experience', 'llm_required_years_of_work_experience', 'salary']),
]:
    for campo in campos:
        p = presente(tabla[campo])
        n = pd.to_numeric(tabla[campo], errors='coerce')
        filas_formato.append({'Entidad': entidad, 'Campo': campo, 'Presentes': int(p.sum()),
                              'Parseables como número': int(n.notna().sum()),
                              'Texto/rango entre los presentes': int((p & n.isna()).sum())})
display(pd.DataFrame(filas_formato))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, tabla, campo, titulo in [
    (axes[0], candidatos_train, 'years_experience', 'Experiencia humana · candidatos'),
    (axes[1], candidatos_train, 'llm_years_of_work_experience', 'Experiencia LLM · candidatos'),
    (axes[2], ofertas_train, 'llm_required_years_of_work_experience', 'Experiencia requerida LLM · ofertas'),
]:
    valores = tabla[campo].fillna('Sin dato').value_counts()
    if campo == 'llm_required_years_of_work_experience':
        valores = valores.reindex(sorted(valores.index, key=lambda x: float(x) if x != 'Sin dato' else np.inf))
    else:
        orden = (['0-2 years', '3-6 years', '+6 years', 'Sin dato'] if campo == 'years_experience'
                 else ['0-2', '2-5', '5-10', '10-15', '15-20', '20-30', '>=30', 'Sin dato'])
        orden = [v for v in orden if v in valores.index] + [v for v in valores.index if v not in orden]
        valores = valores.reindex(orden)
    (valores.iloc[::-1] / len(tabla) * 100).plot.barh(ax=ax, color='#527aa5')
    ax.set(title=titulo, xlabel='Perfiles del subconjunto (%)', ylabel='')
fig.tight_layout()
plt.show()

contratos = pd.DataFrame({
    'Candidatos': candidatos_train['contract_type'].fillna('Sin dato').value_counts(normalize=True),
    'Ofertas': ofertas_train['contract_type'].fillna('Sin dato').value_counts(normalize=True),
}).fillna(0).mul(100)
display(contratos.round(2))

fig, ax = plt.subplots(figsize=(8, 3.5))
salarios_resumen = []
for nombre, tabla, campo, color in [
    ('Salario actual · candidatos', candidatos_train, 'actual_salary', '#2563eb'),
    ('Salario publicado · ofertas', ofertas_train, 'salary', '#d97706'),
]:
    valores = pd.to_numeric(tabla[campo], errors='coerce')
    observados = valores.dropna()
    observados.plot.hist(ax=ax, bins=25, alpha=.45, density=True, label=nombre, color=color)
    salarios_resumen.append({'Campo': nombre, 'Observaciones': len(observados),
                             'Sin dato (%)': 100 * valores.isna().mean(), 'Ceros': observados.eq(0).sum(),
                             'Mediana observada (€ / año)': observados.median(), 'P95 observado': observados.quantile(.95)})
ax.set(title='Salarios disponibles en perfiles con historial de entrenamiento',
       xlabel='Euros anuales', ylabel='Densidad entre valores observados')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
display(pd.DataFrame(salarios_resumen).round(2))


**Interpretación.** Las distribuciones salariales se condicionan a tener un valor observado y corresponden a conceptos diferentes, salario actual y salario de oferta. No comparan directamente expectativas ni permiten tratar ausencia como cero. La tarjeta del dataset informa perturbación y recorte de valores extremos. Los contratos pueden enumerar varias opciones separadas por `;`: la tabla cuenta la combinación completa tal como viene en el CSV, no su compatibilidad semántica. Para implementar contenido, comenzaremos por campos cuyo espacio sea compartido y dejaremos explícito el tratamiento de cada formato.


## 16. ¿La cobertura del contenido es igual entre cohortes?

El subconjunto de entrenamiento contiene muy pocos valores humanos de experiencia requerida para las ofertas. Un porcentaje global del CSV puede esconder este patrón. Comparamos la cobertura de atributos por entidades distintas con procesos en entrenamiento reducido y por entidades distintas con procesos en validación.

Las entidades pueden pertenecer a ambos subconjuntos y los atributos son una instantánea. Esto es una comparación de composición entre cohortes, **no una reconstrucción de cuándo se completó cada perfil**. No ajustamos ninguna transformación con validación.


In [ ]:
cobertura_por_cohorte = []
for nombre, tabla, identificador in [
    ('Candidatos', candidatos, 'candidate_id'), ('Ofertas', ofertas, 'job_id')
]:
    campos = ['skills', 'llm_hard_skills', 'job_category', 'llm_job_category', 'years_experience']
    for cohorte, procesos in [('Train reducido', train_reducido), ('Validación', validacion)]:
        perfiles = tabla[tabla[identificador].isin(procesos[identificador])]
        for campo in campos:
            cobertura_por_cohorte.append({
                'Entidad': nombre, 'Cohorte': cohorte, 'Campo': campo,
                'Perfiles distintos': len(perfiles), 'Cobertura (%)': 100 * presente(perfiles[campo]).mean(),
            })
cobertura_por_cohorte = pd.DataFrame(cobertura_por_cohorte)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.7))
for ax, entidad in zip(axes, ['Candidatos', 'Ofertas']):
    vista = cobertura_por_cohorte[cobertura_por_cohorte['Entidad'] == entidad].pivot(
        index='Campo', columns='Cohorte', values='Cobertura (%)'
    )
    vista.plot.barh(ax=ax, color=['#2563eb', '#d97706'])
    ax.set(title=entidad, xlabel='Perfiles con valor (%)', ylabel='', xlim=(0, 100))
    ax.legend(fontsize=8)
fig.suptitle('Cobertura de atributos según cohorte de interacción', y=1.02)
fig.tight_layout()
plt.show()
display(cobertura_por_cohorte.round(2))


**Consecuencia.** Un campo útil por definición puede aportar poco si casi no existe entre las entidades de entrenamiento. Esta comparación motiva revisar cobertura al evaluar contenido y justificar qué campos se usan. La disponibilidad en validación no garantiza que la distribución de sus valores sea igual a entrenamiento ni que la fuente LLM sea más precisa.


## 17. Habilidades compartidas y representación de contenido

Reutilizamos la selección de campos de `jth.features`, que toma habilidades, herramientas, lenguajes, áreas y categorías. Se excluyen las features demográficas y el marcador `_rare_skill_`.

Para esta EDA contamos **presencia de cada token una sola vez por perfil**, aunque aparezca tanto en una fuente humana como en una LLM. Esto evita confundir repetición con cobertura. El modelo TF-IDF puede tener otra ponderación, que deberá evaluarse explícitamente. Las frecuencias se calculan solo sobre los perfiles con procesos en entrenamiento reducido.


In [ ]:
from jth.features import profile_tokens

tokens_candidatos = pd.Series([sorted(set(t)) for t in profile_tokens(candidatos_train)])
tokens_ofertas = pd.Series([sorted(set(t)) for t in profile_tokens(ofertas_train)])
vocab_c = set(tokens_candidatos.explode().dropna())
vocab_j = set(tokens_ofertas.explode().dropna())

resumen_contenido = pd.DataFrame([
    {'Entidad': nombre, 'Perfiles': len(tokens), 'Sin tokens (%)': 100 * tokens.str.len().eq(0).mean(),
     'Tokens únicos por perfil (mediana)': tokens.str.len().median(), 'Vocabulario distinto': len(vocab)}
    for nombre, tokens, vocab in [
        ('Candidatos', tokens_candidatos, vocab_c), ('Ofertas', tokens_ofertas, vocab_j)
    ]
])
display(resumen_contenido.round(2))
display(pd.DataFrame({
    'Medida de vocabulario': ['Compartido', 'Solo candidatos', 'Solo ofertas', 'Unión'],
    'Tokens': [len(vocab_c & vocab_j), len(vocab_c - vocab_j), len(vocab_j - vocab_c), len(vocab_c | vocab_j)],
}))

habilidades_c = tokens_candidatos.map(lambda xs: [x[2:] for x in xs if x.startswith('s:')]).explode().value_counts()
habilidades_j = tokens_ofertas.map(lambda xs: [x[2:] for x in xs if x.startswith('s:')]).explode().value_counts()
frecuencias_habilidades = pd.DataFrame({
    'Candidatos (%)': habilidades_c / len(tokens_candidatos) * 100,
    'Ofertas (%)': habilidades_j / len(tokens_ofertas) * 100,
    'Perfiles totales': habilidades_c.add(habilidades_j, fill_value=0),
}).fillna(0)
seleccion = frecuencias_habilidades[frecuencias_habilidades['Perfiles totales'] >= 5].copy()
seleccion['Promedio de porcentajes'] = seleccion[['Candidatos (%)', 'Ofertas (%)']].mean(axis=1)
top_habilidades = seleccion.nlargest(12, 'Promedio de porcentajes')
top_habilidades[['Candidatos (%)', 'Ofertas (%)']].sort_values('Candidatos (%)').plot.barh(
    figsize=(10, 5), color=['#2563eb', '#d97706']
)
plt.title('Habilidades frecuentes · presencia por perfil de entrenamiento')
plt.xlabel('Perfiles que contienen la habilidad (%) · multietiqueta')
plt.ylabel('')
plt.tight_layout()
plt.show()


**Consecuencia.** El vocabulario compartido permite construir similitudes entre candidatos y ofertas incluso sin interacciones previas del candidato. Una habilidad frecuente puede ser poco discriminativa; frecuencia no implica poder predictivo. La cobertura y el solapamiento léxico tampoco prueban que el contenido supere a LR. Para un experimento honesto, el vocabulario e IDF aprendidos deberán fijarse usando solo entrenamiento y aplicarse después a perfiles nuevos.


## 18. Composición de los procesos y observación de etapas

Distinguimos **etapa máxima final** y **presencia de una fecha de etapa**. La primera distribución no es un embudo de conversiones y la segunda tampoco implica que todos los procesos recorran las mismas etapas sin saltos.

Analizamos procesos cuyo inicio pertenece al entrenamiento reducido. Sus resultados finales son retrospectivos: algunas etapas pueden ocurrir después del corte de ese entrenamiento. Por eso comparamos fechas presentes en la instantánea con fechas realmente anteriores al corte. Los resultados finales no entran como features al predecir el inicio del proceso.


In [ ]:
mascara_inicio_train = primera_fecha.dt.normalize() < dia_corte
historial_inicio_train = historial.loc[mascara_inicio_train]
etapas_inicio_train = fechas_etapas.loc[mascara_inicio_train]
ETIQUETAS_ETAPA = {
    'Application Made': 'Postulación', 'Shortlist': 'Preselección', 'Qualification': 'Calificación',
    'Resume Sent': 'CV enviado', '1st Interview': '1.ª entrevista', '2nd Interview': '2.ª entrevista',
    '3rd Interview': '3.ª entrevista', '4th Interview': '4.ª entrevista',
    'Offer Received': 'Oferta recibida', 'Offer Accepted': 'Oferta aceptada',
}
maxima = historial_inicio_train['last_stage_reached'].value_counts().reindex(ETIQUETAS_ETAPA, fill_value=0)
maxima.index = [ETIQUETAS_ETAPA[x] for x in maxima.index]
etapas_disponibles = pd.DataFrame({
    'Con fecha en la instantánea (%)': 100 * etapas_inicio_train.notna().mean(),
    'Con fecha anterior al corte (%)': 100 * etapas_inicio_train.lt(dia_corte).mean(),
})
etapas_disponibles.index = list(ETIQUETAS_ETAPA.values())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
(maxima / len(historial_inicio_train) * 100).iloc[::-1].plot.barh(ax=axes[0], color='#527aa5')
axes[0].set(title='Etapa máxima final · retrospectiva', xlabel='Procesos iniciados en train (%)', ylabel='')
etapas_disponibles.iloc[::-1].plot.barh(ax=axes[1], color=['#cbd5e1', '#2563eb'])
axes[1].set(title='Fechas de etapas registradas', xlabel='Procesos iniciados en train (%)', ylabel='')
axes[1].legend(fontsize=8, loc='lower right')
fig.tight_layout()
plt.show()
display(etapas_disponibles.round(2))
print('Etapas de esos procesos fechadas en o después del corte:',
      int(etapas_inicio_train.ge(dia_corte).sum().sum()))


**Implicación.** Si luego representamos una oferta mediante candidatos preseleccionados, solo podremos incorporar a cada candidato cuando la preselección ya haya ocurrido. No basta con que el proceso empezara antes. Una etapa sin fecha no demuestra que el candidato la haya rechazado; puede no haberse registrado o no corresponder a ese proceso. Si cambiamos el positivo a contratación, estaremos definiendo otra tarea y afrontando distinta observabilidad y desbalance.


## 19. Vida activa observada y evolución semanal

La vida activa observada es la diferencia entre el último y el primer **inicio de proceso** de una entidad dentro del entrenamiento reducido. No equivale a la vida real de un candidato en el mercado ni al tiempo de apertura de una oferta.

Las entidades observadas un solo día tienen duración cero con esta definición. Mostramos también la mediana condicionada a dos o más días, y recordamos que la ventana de observación recorta historiales. No utilizamos eventos futuros para construir estas duraciones de entrenamiento.


In [ ]:
vidas = {}
resumen_vidas = []
for nombre, identificador in [('Candidatos', 'candidate_id'), ('Ofertas', 'job_id')]:
    resumen = train_reducido.assign(dia=train_reducido['timestamp'].dt.normalize()).groupby(identificador)['dia'].agg(
        inicio='min', fin='max', dias_distintos='nunique'
    )
    duracion = (resumen['fin'] - resumen['inicio']).dt.days
    vidas[nombre] = duracion
    resumen_vidas.append({
        'Entidad': nombre, 'Entidades observadas': len(resumen),
        'Con actividad en un solo día (%)': 100 * resumen['dias_distintos'].eq(1).mean(),
        'Mediana de duración (días)': duracion.median(), 'P90 (días)': duracion.quantile(.9),
        'Mediana con ≥2 días de actividad': duracion[resumen['dias_distintos'] >= 2].median(),
    })
display(pd.DataFrame(resumen_vidas).round(2))
fig, ax = plt.subplots(figsize=(8.5, 3.6))
for nombre, duracion in vidas.items():
    x = np.sort(duracion.to_numpy())
    ax.step(x, np.arange(1, len(x) + 1) / len(x) * 100, where='post', label=nombre)
ax.set_xscale('symlog', linthresh=1)
ax.set(title='Duración activa observada · entrenamiento reducido',
       xlabel='Días entre primer y último proceso (escala symlog; incluye cero)',
       ylabel='Entidades con duración ≤ x (%)', ylim=(0, 100))
ax.legend()
ax.grid(alpha=.2)
fig.tight_layout()
plt.show()

serie_semanal = train_reducido.groupby(pd.Grouper(key='timestamp', freq='W-SUN')).agg(
    Procesos=('candidate_id', 'size'), Candidatos=('candidate_id', 'nunique'), Ofertas=('job_id', 'nunique')
).fillna(0)
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
serie_semanal[['Procesos']].plot(ax=axes[0], color=['#334155'])
serie_semanal[['Candidatos', 'Ofertas']].plot(ax=axes[1], color=['#2563eb', '#d97706'])
axes[0].set(title='Volumen y entidades con procesos por semana · entrenamiento reducido', ylabel='Procesos')
axes[1].set(xlabel='Semana (etiquetada por su domingo)', ylabel='Entidades distintas')
for ax in axes:
    ax.grid(alpha=.2)
fig.tight_layout()
plt.show()


**Interpretación.** Las curvas permiten ver qué proporción concentra su actividad en un intervalo corto y cuánto pesa la cola larga. En candidatos, una mediana global de cero coexiste con retornos muy separados entre quienes tienen varios días de actividad. El intervalo entre primera y última interacción incluye períodos sin observación y puede abarcar episodios diferentes; no mide una única sesión de búsqueda. Por eso no lo equiparamos a otras definiciones de vida activa de la literatura ni inferimos una fecha de cierre desde la última interacción observada. Las semanas inicial y final del período pueden estar incompletas y las fechas están perturbadas; no atribuimos picos a causas de negocio sin evidencia adicional. La suma de candidatos semanales tampoco equivale a candidatos únicos de todo el período.


## 20. Cold-start al comienzo de cada día de validación

Ahora simulamos únicamente la **disponibilidad del historial**, sin entrenar ni puntuar un modelo. Iniciamos el estado con entrenamiento reducido. Para cada día de validación, construimos las consultas, consultamos el estado anterior y recién después revelamos todos los procesos de ese día.

Conservamos un positivo por candidato–día usando la última fila, como el evaluador oficial. Registramos también el tiempo desde la última interacción previa de su oferta relevante. Las ofertas y candidatos del mismo día no pueden proporcionarse historial entre sí antes de ser evaluados.


In [ ]:
hist_candidatos = set(train_reducido['candidate_id'])
hist_ofertas = set(train_reducido['job_id'])
ultima_actividad_oferta = train_reducido.groupby('job_id')['timestamp'].max().dt.normalize().to_dict()
registros_consultas = []

for dia, procesos_dia in validacion.assign(dia=validacion['timestamp'].dt.normalize()).groupby('dia', sort=True):
    consultas_dia = procesos_dia.drop_duplicates('candidate_id', keep='last').copy()
    ultima_previa = pd.to_datetime(consultas_dia['job_id'].map(ultima_actividad_oferta))
    bloque = pd.DataFrame({
        'dia': dia,
        'candidate_id': consultas_dia['candidate_id'],
        'job_id': consultas_dia['job_id'],
        'candidato_con_historial': consultas_dia['candidate_id'].isin(hist_candidatos),
        'oferta_con_historial': consultas_dia['job_id'].isin(hist_ofertas),
        'dias_desde_ultima_actividad': (dia - ultima_previa).dt.days,
        'fecha_creacion_oferta': consultas_dia['job_id'].map(creacion_ofertas),
    })
    registros_consultas.append(bloque)
    # Solo después de construir TODAS las consultas del día revelamos sus procesos.
    hist_candidatos.update(procesos_dia['candidate_id'])
    hist_ofertas.update(procesos_dia['job_id'])
    ultima_actividad_oferta.update(dict.fromkeys(procesos_dia['job_id'], dia))

consultas_validacion = pd.concat(registros_consultas, ignore_index=True)
assert len(consultas_validacion) == validacion.assign(dia=validacion['timestamp'].dt.normalize()).groupby(['candidate_id', 'dia']).ngroups
assert consultas_validacion.loc[consultas_validacion['oferta_con_historial'], 'dias_desde_ultima_actividad'].ge(1).all()

cohortes = pd.crosstab(
    consultas_validacion['candidato_con_historial'].map({False:'Candidato nuevo', True:'Candidato con historial'}),
    consultas_validacion['oferta_con_historial'].map({False:'Oferta nueva en historial', True:'Oferta con historial'}),
).reindex(index=['Candidato nuevo', 'Candidato con historial'],
          columns=['Oferta nueva en historial', 'Oferta con historial'], fill_value=0)
porcentajes_cohortes = cohortes / len(consultas_validacion) * 100
display(cohortes)
fig, ax = plt.subplots(figsize=(7.5, 3.2))
im = ax.imshow(porcentajes_cohortes.to_numpy(), cmap='Blues', vmin=0, vmax=100)
ax.set_xticks(range(2), ['Oferta nueva\nen historial', 'Oferta con\nhistorial'])
ax.set_yticks(range(2), ['Candidato\nnuevo', 'Candidato con\nhistorial'])
for i in range(2):
    for j in range(2):
        v = porcentajes_cohortes.iloc[i, j]
        ax.text(j, i, f'{cohortes.iloc[i, j]:,} consultas\n{v:.1f}%',
                ha='center', va='center', color='white' if v > 50 else '#111827')
ax.set_title('Cuatro escenarios de historial previo · consultas de validación')
fig.colorbar(im, ax=ax, label='Porcentaje de todas las consultas')
fig.tight_layout()
plt.show()

comparacion_nuevos = []
for entidad, clave, bandera in [
    ('Candidatos', 'candidate_id', 'candidato_con_historial'),
    ('Ofertas', 'job_id', 'oferta_con_historial'),
]:
    unicas_val = validacion[clave].drop_duplicates()
    comparacion_nuevos.append({
        'Entidad': entidad,
        'Entidades distintas de val ausentes en train reducido (%)': 100 * (~unicas_val.isin(train_reducido[clave])).mean(),
        'Consultas sin historial al recomendar (%)': 100 * (~consultas_validacion[bandera]).mean(),
    })
display(pd.DataFrame(comparacion_nuevos).round(2))
cohortes_mensuales = consultas_validacion.set_index('dia').resample('MS').agg(
    consultas=('candidate_id', 'size'), candidato_warm=('candidato_con_historial', 'mean'),
    oferta_warm=('oferta_con_historial', 'mean')
)
ax = cohortes_mensuales[['candidato_warm', 'oferta_warm']].mul(100).rename(columns={
    'candidato_warm':'Candidato con historial', 'oferta_warm':'Oferta con historial'
}).plot(marker='o', figsize=(9, 3.5))
ax.set(title='Historial disponible por mes · validación', xlabel='Mes', ylabel='Consultas (%)', ylim=(0, 100))
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()


**Implicación para el objetivo.** Un candidato ausente del entrenamiento puede adquirir historial durante validación. Los porcentajes por entidades distintas y por consultas dinámicas tienen denominadores diferentes. La combinación de candidatos nuevos con ofertas que sí tienen historial es el escenario que motiva el perfil dinámico centrado en ofertas; no demuestra todavía una mejora de ranking.


## 21. Tiempo desde creación frente a tiempo desde última actividad

La edad desde publicación y el tiempo desde la última interacción no son la misma señal. Una oferta antigua puede seguir recibiendo candidatos. Comparamos ambas distribuciones en los positivos de validación y mostramos por separado las ofertas sin historial previo.

El cálculo usa información anterior al día de cada consulta. La fecha de creación procede de la instantánea de perfiles; si el positivo aparece antes de esa fecha, se informa como inconsistencia de elegibilidad en lugar de convertir su edad en cero.


In [ ]:
consultas_validacion['edad_desde_creacion'] = (
    consultas_validacion['dia'] - consultas_validacion['fecha_creacion_oferta']
).dt.days
def descripcion_tiempo(nombre, serie):
    validos = serie[serie.notna() & serie.ge(0)]
    return {'Medida': nombre, 'Consultas con valor válido': len(validos),
            'Sin valor': int(serie.isna().sum()), 'Valores negativos': int(serie.lt(0).sum()),
            'Mediana (días)': validos.median(), 'P90 (días)': validos.quantile(.9)}
display(pd.DataFrame([
    descripcion_tiempo('Edad desde creación', consultas_validacion['edad_desde_creacion']),
    descripcion_tiempo('Tiempo desde actividad previa', consultas_validacion['dias_desde_ultima_actividad']),
]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, nombre, serie in [
    (axes[0], 'Desde creación', consultas_validacion['edad_desde_creacion']),
    (axes[1], 'Desde última actividad previa', consultas_validacion['dias_desde_ultima_actividad']),
]:
    validos = np.sort(serie[serie.notna() & serie.ge(0)].to_numpy())
    if len(validos):
        ax.step(validos, np.arange(1, len(validos)+1) / len(validos) * 100, where='post', color='#2563eb')
    ax.set_xscale('symlog', linthresh=1)
    ax.set(title=nombre, xlabel='Días (escala symlog; incluye cero)',
           ylabel='Consultas entre las que tienen valor válido (%)', ylim=(0, 100))
    ax.grid(alpha=.2)
fig.suptitle('Ofertas relevantes de validación · dos nociones de recencia', y=1.02)
fig.tight_layout()
plt.show()


**Interpretación.** Las curvas están condicionadas a disponer de cada valor; no debemos compararlas como si incluyeran siempre las mismas consultas. La tabla indica qué casos quedaron fuera. Para estudiar un filtro duro, la siguiente sección mantendrá **todas** las consultas en el denominador y contará como pérdidas las ofertas relevantes que no entren al pool.


## 22. Cobertura de ventanas de actividad y tamaño del catálogo

Esta es una exploración de recuperación de candidatos, no una evaluación de MRR. Para cada día de validación tomamos las ofertas con fecha de creación conocida y no futura. Dentro de ese catálogo identificamos las que tuvieron al menos un proceso en los `w` días anteriores, excluyendo el día de consulta.

Probamos ventanas descriptivas de 7, 14, 30, 60 y 90 días sin declarar una ganadora. Reportamos cobertura del positivo sobre las **4.208 consultas**, pérdidas y tamaño del pool. Los tamaños se resumen por día con consultas; la cobertura se pondera por consulta. Son denominadores distintos, expresamente identificados.

No protegemos aquí ofertas recién publicadas sin historial: queremos medir qué perdería la regla basada únicamente en actividad. Diseñar ese fallback será un paso posterior, antes de implementar el filtro.


In [ ]:
VENTANAS_EXPLORACION = [7, 14, 30, 60, 90]
historia_para_validacion = pd.concat([train_reducido, validacion], ignore_index=True).assign(
    dia=lambda df: df['timestamp'].dt.normalize()
)
filas_ventanas = []
for dia, consultas_dia in consultas_validacion.groupby('dia', sort=True):
    pasado = historia_para_validacion[historia_para_validacion['dia'] < dia]
    ultima = pasado.groupby('job_id')['dia'].max()
    elegibles = set(creacion_ofertas.index[creacion_ofertas.notna() & creacion_ofertas.le(dia)])
    for ventana in VENTANAS_EXPLORACION:
        activas = set(ultima.index[ultima.ge(dia - pd.Timedelta(days=ventana))]) & elegibles
        filas_ventanas.append({
            'Día': dia, 'Ventana (días)': ventana, 'Consultas': len(consultas_dia),
            'Positivos conservados': int(consultas_dia['job_id'].isin(activas).sum()),
            'Ofertas elegibles': len(elegibles), 'Ofertas activas elegibles': len(activas),
            'Fracción del catálogo (%)': 100 * len(activas) / len(elegibles) if elegibles else np.nan,
        })
ventanas_por_dia = pd.DataFrame(filas_ventanas)
resumen_ventanas = ventanas_por_dia.groupby('Ventana (días)').agg(
    consultas=('Consultas', 'sum'), conservados=('Positivos conservados', 'sum'),
    ofertas_pool_mediana=('Ofertas activas elegibles', 'median'),
    catalogo_mediana=('Ofertas elegibles', 'median'), fraccion_catalogo_mediana=('Fracción del catálogo (%)', 'median'),
)
resumen_ventanas['Cobertura por consulta (%)'] = 100 * resumen_ventanas['conservados'] / resumen_ventanas['consultas']
resumen_ventanas['Consultas cuyo positivo se elimina'] = resumen_ventanas['consultas'] - resumen_ventanas['conservados']
assert resumen_ventanas['consultas'].eq(len(consultas_validacion)).all()
assert resumen_ventanas['conservados'].is_monotonic_increasing
display(resumen_ventanas.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
resumen_ventanas[['Cobertura por consulta (%)', 'fraccion_catalogo_mediana']].rename(columns={
    'fraccion_catalogo_mediana':'Catálogo conservado · mediana por día (%)'
}).plot(ax=axes[0], marker='o')
axes[0].set(title='Cobertura y fracción de catálogo', ylabel='Porcentaje', ylim=(0, 100))
resumen_ventanas[['ofertas_pool_mediana']].rename(columns={
    'ofertas_pool_mediana':'Pool activo · mediana por día'
}).plot(ax=axes[1], marker='o', color=['#d97706'])
axes[1].set(title='Cantidad de ofertas que habría que ordenar', ylabel='Ofertas', ylim=(0, None))
for ax in axes:
    ax.set_xlabel('Ventana de actividad previa (días)')
    ax.set_xticks(VENTANAS_EXPLORACION)
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

nuevo_sin_historial = ~consultas_validacion['oferta_con_historial']
display(pd.DataFrame({
    'Casos que requieren atención': ['Oferta relevante sin historial previo',
                                  'Oferta relevante con creación posterior a la consulta'],
    'Consultas de validación': [int(nuevo_sin_historial.sum()),
                              int(consultas_validacion['edad_desde_creacion'].lt(0).sum())],
}))


**Qué permite concluir.** Una ventana corta puede reducir mucho el catálogo y también perder positivos. Si el filtro es irreversible, esos casos no pueden recuperarse en el ranking y deben seguir contando como fallos. La cobertura no equivale a precisión ni MRR, y el tamaño del pool no mide por sí solo tiempo de ejecución. Ninguna curva demuestra que una oferta siga realmente abierta.

**Decisión que sigue pendiente.** Comparar filtros duros con pesos suaves y definir una regla para ofertas nuevas. No elegimos `w=30` porque aparezca en la propuesta; la tabla de validación servirá para justificar qué configuraciones limitadas probar junto a un modelo y con incertidumbre.

La cobertura calculada aquí usa consultas de validación y una regla explícita de catálogo. No debe sustituirse por el 92 % citado en la propuesta, que describía procesos del test. Una diferencia entre esos porcentajes no cuantifica por sí sola un cambio temporal: también difieren el período, la unidad y el filtro aplicado.


## 23. Primer control de Recencia y preparación de contenido

**Registro de la primera etapa, opción A.** Ejecutamos Recencia en las 4.208 consultas de validación, entendemos sus métricas y preparamos el siguiente modelo. Todavía no ajustamos ni evaluamos TF-IDF con datos reales.

Este bloque utiliza solo variables preparadas en los puntos 1–9. Toda la explicación y los resultados quedan en este notebook. Los programas oficiales corren desde `scripts/run_references.py`, y sus resultados quedan en la caché ignorada por git.

| Componente | Decisión para este control |
|---|---|
| Pregunta | ¿Cuánto consigue ordenar únicamente por la fecha de publicación? |
| Datos evaluados | Validación elegida dentro del entrenamiento oficial |
| Generador de ranking | `external/JTH/code/baselines/past_temporal_baseline.py`, sin modificar |
| Evaluación | Función `compute_metrics` del evaluador oficial, sin modificar |
| Lista | Hasta 1.000 ofertas; el evaluador excluye las creadas después de la consulta |
| Aleatoriedad | Semilla 42 para los desempates del generador y las métricas auxiliares |
| Métricas principales | MRR y nDCG@10, global y por historial previo; HitRate@10 como ayuda interpretativa |

Estos números serán de **validación**. No se comparan directamente con resultados publicados sobre otro período de test. Recencia no necesita entrenamiento estadístico: no aprende preferencias ni usa los atributos del candidato.

**Actualización:** este punto conserva el estado al cerrar Recencia. Las secciones 24–27 completan ahora contenido, referencias, actividad y fusión.


### 23.1. Entender la regla con un ejemplo inventado

Al consultar en un día, Recencia prioriza ofertas ya publicadas de menor antigüedad. Aquí el positivo inventado es `oferta_A`, que es elegible pero más antigua que otras dos ofertas. La oferta futura queda fuera del ranking evaluado.

El código oficial incluye un tratamiento de desempates con semilla. Sin empates, dos candidatos consultados el mismo día reciben el mismo orden básico, porque esta regla no personaliza.


In [ ]:
import json
import random
import time
import importlib.util
from IPython.display import Markdown

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

dia_toy_recencia = pd.Timestamp('2020-01-10')
ejemplo_recencia = pd.DataFrame({
    'Oferta inventada': ['oferta_A', 'oferta_B', 'oferta_C', 'oferta_futura'],
    'Publicada': pd.to_datetime(['2020-01-01', '2020-01-05', '2020-01-09', '2020-01-12']),
    'Es el positivo': [True, False, False, False],
})
ejemplo_recencia['Elegible'] = ejemplo_recencia['Publicada'].le(dia_toy_recencia)
ejemplo_recencia['Antigüedad (días)'] = (dia_toy_recencia - ejemplo_recencia['Publicada']).dt.days
ejemplo_recencia = ejemplo_recencia.sort_values(['Elegible', 'Publicada'], ascending=[False, False])
ejemplo_recencia['Puesto evaluado'] = pd.Series(pd.NA, index=ejemplo_recencia.index, dtype='Int64')
ejemplo_recencia.loc[ejemplo_recencia['Elegible'], 'Puesto evaluado'] = np.arange(1, 4)
display(ejemplo_recencia.reset_index(drop=True))
assert ejemplo_recencia.loc[ejemplo_recencia['Es el positivo'], 'Puesto evaluado'].iloc[0] == 3


**Observación.** La oferta correcta queda tercera aunque podría ser la más afín al candidato. Recencia funciona como control de señal temporal, no como prueba de afinidad. Fecha reciente tampoco confirma que una oferta esté abierta.

### 23.2. Qué aportan las métricas en una consulta

Con un positivo por consulta, su aporte a MRR es `1 / puesto`. Para nDCG@10 es `1 / log2(puesto + 1)` si aparece entre las diez primeras, y cero en caso contrario. HitRate@10 vale uno si aparece entre esas diez. En este protocolo de un positivo, coincide con Recall@10.

Si el positivo no está en la lista top-1000, su aporte al MRR evaluado es cero. MRR no es el inverso del puesto promedio: primero calculamos el recíproco de cada consulta y luego promediamos.


In [ ]:
puestos_ejemplo = np.array([1., 3., 10., 11., np.inf])
tabla_metricas_ejemplo = pd.DataFrame({
    'Caso': ['Puesto 1', 'Puesto 3', 'Puesto 10', 'Puesto 11', 'Ausente de la lista top-1000'],
    'Aporte a MRR': 1 / puestos_ejemplo,
    'Aporte a nDCG@10': np.where(puestos_ejemplo <= 10, 1 / np.log2(puestos_ejemplo + 1), 0),
    'Aporte a HitRate@10': (puestos_ejemplo <= 10).astype(float),
})
display(tabla_metricas_ejemplo.round(4))
assert np.isclose(tabla_metricas_ejemplo.loc[1, 'Aporte a MRR'], 1/3)
assert tabla_metricas_ejemplo.loc[1, 'Aporte a nDCG@10'] == .5
assert tabla_metricas_ejemplo.loc[4, 'Aporte a MRR'] == 0


### 23.3. Verificar el evaluador oficial con casos inventados

Importamos las funciones oficiales sin ejecutar su interfaz de comandos. Comprobamos tres consultas inventadas: una con positivo tercero después de eliminar una oferta futura, otra con positivo primero y otra sin predicción. La última debe seguir contando y aportar cero.

El candidato A aparece de nuevo al día siguiente y pasa de cold a warm. Esta prueba conecta las fórmulas anteriores con el código que usaremos en datos reales.


In [ ]:
RUTA_EVALUADOR = RAIZ / 'external/JTH/code/utils/evaluator.py'
RUTA_RECENCIA = RAIZ / 'external/JTH/code/baselines/past_temporal_baseline.py'
from jth.context import load_official_evaluator
evaluador_oficial = load_official_evaluator(RAIZ)

gold_toy = {
    ('candidato_A', '2020-01-10'): 'oferta_A',
    ('candidato_A', '2020-01-11'): 'oferta_B',
    ('candidato_B', '2020-01-11'): 'oferta_C',
}
preds_toy = {
    ('candidato_A', '2020-01-10'): ['oferta_futura', 'oferta_C', 'oferta_B', 'oferta_A'],
    ('candidato_A', '2020-01-11'): ['oferta_B'],
}
fechas_toy = {'oferta_A':'2020-01-01', 'oferta_B':'2020-01-05',
             'oferta_C':'2020-01-09', 'oferta_futura':'2020-01-12'}
random.seed(42)
tablas_toy, _, mrr_toy, n_toy = evaluador_oficial.compute_metrics(
    gold_toy, preds_toy, [10], fechas_toy, set()
)
assert n_toy == {'all':3, 'warm':1, 'cold':2}
assert np.isclose(mrr_toy['all'], (1/3 + 1 + 0) / 3)
assert np.isclose(tablas_toy['all'].iloc[0]['NDCG@K'], .5)
display(pd.DataFrame([
    {'Grupo inventado': grupo, 'Consultas': n_toy[grupo], 'MRR': mrr_toy[grupo]}
    for grupo in ['all', 'warm', 'cold']
]))
print('Pruebas sintéticas: positivo tercero, futuro excluido, predicción ausente y cambio cold→warm: OK.')


### 23.4. Ejecutar Recencia oficial sobre toda la validación

Guardamos temporalmente el entrenamiento reducido y la validación con las tres columnas esperadas. El generador oficial recibe perfiles y consultas de validación; **no recibe los positivos para aprender sus scores**. El archivo de consultas incluye la oferta para poder evaluarla después, pero la regla de Recencia solo usa candidato, fecha y fechas de creación de ofertas.

El generador procesa las filas y su diccionario conserva una lista por candidato–día, incluso si hay varios procesos. El evaluador carga el positivo según su propia regla de sobrescritura. Medimos por separado generación y evaluación. El archivo de ranking se carga en memoria antes de eliminar la carpeta temporal.


In [ ]:
SEMILLA_RECENCIA = 42
KS_RECENCIA = [1, 5, 10, 20]

# Corre todas las referencias oficiales de una vez, o reutiliza su caché verificada.
spec_runner = importlib.util.spec_from_file_location('run_references', RAIZ / 'scripts' / 'run_references.py')
runner_referencias = importlib.util.module_from_spec(spec_runner)
spec_runner.loader.exec_module(runner_referencias)
cache_referencias = runner_referencias.execute(RAIZ, 'val')

ruta_train_r = cache_referencias / 'train.csv'
ruta_val_r = cache_referencias / 'target.csv'
ruta_preds_r = cache_referencias / 'recency' / 'recency_c2j.pkl.gz'
pd.testing.assert_frame_equal(pd.read_csv(ruta_train_r, dtype=str, parse_dates=['timestamp']), train_reducido.reset_index(drop=True))
pd.testing.assert_frame_equal(pd.read_csv(ruta_val_r, dtype=str, parse_dates=['timestamp']), validacion.reset_index(drop=True))

gold_recencia = evaluador_oficial.load_gold(ruta_val_r, 'c2j')
preds_recencia = evaluador_oficial.load_preds(ruta_preds_r)
fechas_objetivo_recencia = evaluador_oficial.load_dates(RAW/'jobs.csv', 'job_id')
conocidos_iniciales_recencia = evaluador_oficial.load_initial_known_queries(ruta_train_r, 'c2j')

random.seed(SEMILLA_RECENCIA)
tablas_recencia, auxiliares_recencia, mrr_recencia, n_recencia = evaluador_oficial.compute_metrics(
    gold_recencia, preds_recencia, KS_RECENCIA, fechas_objetivo_recencia, conocidos_iniciales_recencia
)

assert len(gold_recencia) == len(preds_recencia) == 4208
assert set(gold_recencia) == set(preds_recencia)
assert all(len(ranking) <= 1000 and len(ranking) == len(set(ranking)) for ranking in preds_recencia.values())
display(pd.DataFrame({
    'Comprobación': ['Procesos de validación', 'Consultas evaluadas', 'Rankings generados'],
    'Valor': [len(validacion), n_recencia['all'], len(preds_recencia)],
}))
print('Generador y evaluador oficiales ejecutados, con identidad de entrenamiento y validación verificada.')


### 23.5. Reconstruir el rango de cada consulta y comprobar las métricas

Para analizar errores necesitamos una fila por consulta. Reproducimos explícitamente el filtrado del evaluador: una oferta con creación futura se omite y no ocupa posición; una fecha desconocida se mantiene según su regla. Ordenamos las consultas cronológicamente y marcamos el historial del candidato.

Esta tabla es un auxiliar de análisis, no un reemplazo del evaluador oficial. Comprobamos que sus medias y conteos coincidan con el resultado oficial global y por cohorte. No imprimimos IDs ni perfiles de personas reales.


In [ ]:
conocidos_analisis_r = set(conocidos_iniciales_recencia)
filas_analisis_r = []
for clave in sorted(gold_recencia, key=lambda x: x[1]):
    candidato_id, fecha = clave
    positivo = gold_recencia[clave]
    lista_bruta = preds_recencia.get(clave, [])
    lista_elegible = [oferta_id for oferta_id in lista_bruta
                      if not fechas_objetivo_recencia.get(oferta_id, '')
                      or fechas_objetivo_recencia[oferta_id] <= fecha]
    puesto = next((i+1 for i, oferta_id in enumerate(lista_elegible) if oferta_id == positivo), np.inf)
    fecha_positivo = fechas_objetivo_recencia.get(positivo, '')
    filas_analisis_r.append({
        'dia': pd.Timestamp(fecha), 'warm': candidato_id in conocidos_analisis_r,
        'puesto': puesto, 'RR': 1/puesto,
        'nDCG@10': 1/np.log2(puesto+1) if puesto <= 10 else 0.,
        'HitRate@10': float(puesto <= 10), 'largo_elegible': len(lista_elegible),
        'edad_positivo': (pd.Timestamp(fecha)-pd.Timestamp(fecha_positivo)).days if fecha_positivo else np.nan,
    })
    conocidos_analisis_r.add(candidato_id)
detalle_recencia = pd.DataFrame(filas_analisis_r)

filas_resumen_r = []
for grupo, mascara in [('all', pd.Series(True, index=detalle_recencia.index)),
                       ('warm', detalle_recencia['warm']), ('cold', ~detalle_recencia['warm'])]:
    parte = detalle_recencia[mascara]
    tabla_oficial = tablas_recencia[grupo].set_index('K')
    assert len(parte) == n_recencia[grupo]
    assert np.isclose(parte['RR'].mean(), mrr_recencia[grupo], atol=1e-12, rtol=0)
    assert np.isclose(parte['nDCG@10'].mean(), tabla_oficial.loc[10, 'NDCG@K'], atol=1e-12, rtol=0)
    assert np.isclose(parte['HitRate@10'].mean(), tabla_oficial.loc[10, 'HitRate@K'], atol=1e-12, rtol=0)
    filas_resumen_r.append({
        'Grupo': {'all':'Todos', 'warm':'Con historial', 'cold':'Nuevos'}[grupo],
        'Consultas': len(parte), 'MRR': mrr_recencia[grupo],
        'nDCG@10': tabla_oficial.loc[10, 'NDCG@K'], 'HitRate@10': tabla_oficial.loc[10, 'HitRate@K'],
        'Positivo fuera de lista (%)': 100 * np.isinf(parte['puesto']).mean(),
    })
resumen_recencia = pd.DataFrame(filas_resumen_r)
display(resumen_recencia.round(6))
print('Coincidencia del análisis por consulta con el evaluador oficial: OK en todos los grupos.')


**Cómo leer la tabla.** MRR y nDCG son promedios de aportes, no porcentajes de contrataciones. HitRate@10 expresa la fracción de consultas cuyo positivo aparece entre las diez primeras. El positivo ausente de la lista aporta cero y permanece en el denominador.

Si Recencia rinde diferente entre warm y cold, no significa que haya aprendido a personalizar: usa la misma regla. Las cohortes pueden diferir en días consultados y antigüedad de sus ofertas relevantes. No atribuimos esas diferencias a una causa sin un experimento controlado.

**Lectura de la ejecución guardada.** La lista contiene el positivo en aproximadamente el 99,6 % de las consultas, pero solo el 10,1 % lo sitúa entre las diez primeras. El principal espacio de mejora de este control está en ordenar mejor las ofertas recuperadas. Esto no prueba que contenido lo consiga; esa será la comparación siguiente. La interpretación corresponde a la validación y semilla indicadas en este bloque.


### 23.6. Distribución de puestos y resultados a lo largo del período

La media no muestra cómo se reparten aciertos y errores. Graficamos la posición del positivo, la fracción recuperada hasta distintos K y el MRR mensual. Todos los casos, incluidos los positivos ausentes, forman parte del denominador.

El MRR global se calcula por consulta. No debe obtenerse promediando los MRR mensuales sin ponderarlos por su número de consultas. Los meses extremos son parciales; las curvas son descriptivas, sin intervalos de confianza en este primer control.


In [ ]:
etiquetas_puestos_r = ['1', '2–5', '6–10', '11–100', '101–1000', 'Fuera de lista']
bandas_puesto_r = pd.cut(detalle_recencia['puesto'], [0,1,5,10,100,1000,np.inf], labels=etiquetas_puestos_r)
distribucion_puestos_r = bandas_puesto_r.value_counts(sort=False)
assert distribucion_puestos_r.sum() == len(detalle_recencia)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
(distribucion_puestos_r / len(detalle_recencia) * 100).plot.bar(ax=axes[0], color='#527aa5', rot=20)
axes[0].set(title='¿Dónde queda la oferta relevante?', xlabel='Puesto evaluado', ylabel='Consultas (%)')
valores_k_r = [1,5,10,20,50,100,500,1000]
axes[1].plot(valores_k_r, [100 * detalle_recencia['puesto'].le(k).mean() for k in valores_k_r], marker='o')
axes[1].set_xscale('log')
axes[1].set(title='Positivos recuperados hasta K', xlabel='K (escala logarítmica)', ylabel='Consultas con acierto (%)', ylim=(0,100))
for ax in axes:
    ax.grid(alpha=.2)
fig.tight_layout()
plt.show()

mensual_recencia = detalle_recencia.set_index('dia').resample('MS').agg(
    consultas=('RR','size'), MRR=('RR','mean'), NDCG10=('nDCG@10','mean'), HR10=('HitRate@10','mean')
)
display(mensual_recencia.round(4))
assert np.isclose(np.average(mensual_recencia['MRR'], weights=mensual_recencia['consultas']), mrr_recencia['all'])
ax = mensual_recencia[['MRR','NDCG10']].rename(columns={'NDCG10':'nDCG@10'}).plot(marker='o', figsize=(9,3.4))
ax.set(title='Recencia por mes de validación', xlabel='Mes', ylabel='Métrica promedio')
ax.set_ylim(bottom=0)
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()


### 23.7. Relacionar los resultados con la regla del modelo

Agrupamos por antigüedad de la oferta relevante al consultar. Es una característica descriptiva del resultado observado, no una variable usada para elegir qué consulta evaluar. Los casos fuera de la lista siguen aportando cero.

Si las ofertas antiguas se recuperan peor, es coherente con ordenar por publicación, pero no demuestra por sí solo qué método las rescatará. Las diferencias entre grupos no son una prueba causal y se deben leer junto a sus tamaños.


In [ ]:
bandas_edad_r = pd.cut(
    detalle_recencia['edad_positivo'], [-1,7,14,30,60,90,np.inf],
    labels=['0–7 días','8–14 días','15–30 días','31–60 días','61–90 días','91+ días']
)
por_edad_recencia = detalle_recencia.assign(antiguedad=bandas_edad_r).groupby('antiguedad', observed=False).agg(
    consultas=('RR','size'), MRR=('RR','mean'), HR10=('HitRate@10','mean')
)
assert por_edad_recencia['consultas'].sum() == len(detalle_recencia)
display(por_edad_recencia.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11,3.8))
por_edad_recencia['consultas'].plot.bar(ax=axes[0], color='#94a3b8', rot=25)
axes[0].set(title='Cantidad de consultas por antigüedad del positivo', xlabel='', ylabel='Consultas')
por_edad_recencia['HR10'].mul(100).plot.bar(ax=axes[1], color='#2563eb', rot=25)
axes[1].set(title='Aciertos top-10 por antigüedad', xlabel='', ylabel='Consultas del grupo (%)', ylim=(0,100))
fig.tight_layout()
plt.show()

fila_global_r = resumen_recencia.iloc[0]
display(Markdown(
    f"**Resultado calculado en esta ejecución:** MRR **{fila_global_r['MRR']:.4f}**, "
    f"nDCG@10 **{fila_global_r['nDCG@10']:.4f}** y acierto top-10 en "
    f"**{100*fila_global_r['HitRate@10']:.1f}%** de las **{int(fila_global_r['Consultas']):,} consultas**. "
    "Estas métricas describen validación; no son una réplica del resultado de test del paper."
))


**Observaciones para el proyecto.** Recencia establece cuánto consigue una señal temporal sencilla sin personalización. Puede priorizar ofertas recientes poco afines y relegar ofertas antiguas todavía activas. El filtro por última actividad de la propuesta es una señal distinta de la fecha de publicación: podría conservar ofertas antiguas que siguen recibiendo procesos.

El siguiente modelo de contenido deberá demostrar qué aporta la afinidad de atributos bajo estas mismas consultas y reglas. Los resultados anteriores no justifican cambiar el positivo, retirar consultas ni ajustar parámetros con test.

### 23.8. Preparar el corpus de contenido y el ajuste TF-IDF

Reutilizamos los campos de `profile_tokens`. Como elección inicial conservadora, el corpus que podrá ajustar IDF contiene **perfiles de entidades con al menos un proceso en entrenamiento reducido**. No incluye entidades que aparecen solo en validación. Los atributos son la instantánea publicada; no disponemos de sus versiones históricas y declaramos esa limitación.

Este ajuste es diferente de la implementación histórica, que ajusta IDF con todos los perfiles. Conservamos esa clase original y preparamos aquí una variante temporalmente restringida para el siguiente experimento.

La celda crea documentos y configura el vectorizador, pero **no llama a `fit`, `fit_transform`, `transform` ni calcula rankings de contenido**. Hacerlo será el siguiente paso elegido explícitamente.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from jth.features import profile_tokens, TOKEN_FIELDS

ids_candidatos_fit = set(train_reducido['candidate_id'])
ids_ofertas_fit = set(train_reducido['job_id'])
candidatos_fit_contenido = candidatos[candidatos['candidate_id'].isin(ids_candidatos_fit)].copy()
ofertas_fit_contenido = ofertas[ofertas['job_id'].isin(ids_ofertas_fit)].copy()
docs_candidatos_fit = profile_tokens(candidatos_fit_contenido)
docs_ofertas_fit = profile_tokens(ofertas_fit_contenido)
docs_fit_contenido = docs_candidatos_fit + docs_ofertas_fit

assert set(candidatos_fit_contenido['candidate_id']) == ids_candidatos_fit
assert set(ofertas_fit_contenido['job_id']) == ids_ofertas_fit
assert not {'llm_sex','llm_nationality','llm_age_bucket'}.intersection(TOKEN_FIELDS)

def identidad_tokens_contenido(tokens):
    return tokens

vectorizador_contenido_preparado = TfidfVectorizer(
    analyzer=identidad_tokens_contenido, sublinear_tf=True, norm='l2'
)
assert not hasattr(vectorizador_contenido_preparado, 'vocabulary_')

display(pd.DataFrame([
    {'Corpus preparado': nombre, 'Documentos': len(docs),
     'Documentos vacíos': sum(not doc for doc in docs)}
    for nombre, docs in [('Candidatos de train reducido', docs_candidatos_fit),
                        ('Ofertas de train reducido', docs_ofertas_fit), ('Total para futuro ajuste', docs_fit_contenido)]
]))
display(pd.DataFrame({'Campo seleccionado':list(TOKEN_FIELDS), 'Espacio de tokens':list(TOKEN_FIELDS.values())}))
print('Estado: corpus preparado y vectorizador SIN ajustar. No hay métricas de contenido todavía.')


**Qué aprenderá TF-IDF.** El vocabulario determina qué tokens conoce y el IDF cuánto discrimina cada token en el corpus de ajuste. Después transformaremos los perfiles de consulta y las ofertas disponibles usando ese vocabulario congelado. Los tokens nuevos no vistos durante el ajuste no crearán dimensiones nuevas.

Con normalización L2, el producto entre vectores corresponde a similitud coseno para vectores no nulos. Un perfil sin tokens conocidos produce un vector cero y puede generar muchos empates. Debemos registrar ese caso y fijar un desempate reproducible, sin introducir silenciosamente otra señal temporal.

### 23.9. Preparar la salida compatible con el evaluador oficial

El futuro ranking de contenido se entregará como diccionario de claves JSON `[candidato, fecha]` y listas ordenadas de hasta 1.000 IDs. La función siguiente solo valida y construye ese formato; todavía no recibe predicciones reales de contenido.

Verificamos la ida y vuelta con IDs inventados y el lector oficial. Los rankings reales se calcularán sobre ofertas elegibles, se truncarán con una política de empate explícita y se evaluarán con el mismo `gold_recencia`, fechas y candidatos conocidos que este control.


In [ ]:
import gzip
import pickle

def preparar_formato_oficial(filas_rankings):
    salida = {}
    for candidato_id, dia, ranking in filas_rankings:
        ids = [str(oferta_id) for oferta_id in ranking]
        if len(ids) > 1000 or len(ids) != len(set(ids)):
            raise ValueError('Cada ranking debe contener hasta 1000 IDs distintos y ordenados.')
        clave = json.dumps([str(candidato_id), pd.Timestamp(dia).strftime('%Y-%m-%d')])
        if clave in salida:
            raise ValueError('Hay más de un ranking para la misma consulta candidato-día.')
        salida[clave] = ids
    return salida

ejemplo_formato = preparar_formato_oficial([
    ('candidato_demo', '2020-01-10', ['oferta_A','oferta_B'])
])
with tempfile.TemporaryDirectory(prefix='jth_formato_toy_') as carpeta:
    ruta_toy = Path(carpeta)/'contenido_toy_c2j.pkl.gz'
    with gzip.open(ruta_toy, 'wb') as archivo:
        pickle.dump(ejemplo_formato, archivo)
    lectura_toy = evaluador_oficial.load_preds(ruta_toy)
assert lectura_toy[('candidato_demo','2020-01-10')].tolist() == ['oferta_A','oferta_B']

for caso_invalido in [
    [('candidato_demo','2020-01-10',['oferta_A','oferta_A'])],
    [('candidato_demo','2020-01-10',[]), ('candidato_demo','2020-01-10',[])],
]:
    try:
        preparar_formato_oficial(caso_invalido)
    except ValueError:
        pass
    else:
        raise AssertionError('Debía rechazarse un ranking o una consulta duplicada.')

display(pd.DataFrame({'Clave JSON inventada':list(ejemplo_formato),
                      'Ranking inventado':list(ejemplo_formato.values())}))
print('Formato oficial y rechazo de duplicados comprobados con datos inventados.')


### 23.10. Registrar procedencia y dejar claro qué falta

Los hashes identifican los archivos usados, la semilla y las cantidades identifican el experimento, y los resultados quedan guardados en las salidas de este notebook. Los tiempos corresponden a esta máquina y ejecución.

No se utilizaron las métricas auxiliares de comparación con un negativo aleatorio para decidir nada. Mantenemos MRR, nDCG@10 y aciertos top-10 como indicadores interpretables de este control.


In [ ]:
procedencia_recencia = pd.DataFrame([
    {'Elemento':str(ruta.relative_to(RAIZ)), 'SHA256':hashlib.sha256(ruta.read_bytes()).hexdigest()}
    for ruta in [RUTA_RECENCIA, RUTA_EVALUADOR, RAW/'candidates.csv', RAW/'jobs.csv', RAW/'history.csv']
])
display(procedencia_recencia)
estado_bloque_modelos = pd.DataFrame({
    'Trabajo': ['Ejemplo y comprobación de métricas', 'Recencia oficial en validación',
               'Análisis por consulta y cohorte', 'Corpus y formato de contenido',
               'Ajuste TF-IDF con datos reales', 'Ranking y evaluación de contenido'],
    'Estado': ['Completado', 'Completado', 'Completado', 'Preparado', 'Pendiente', 'Pendiente'],
})
display(estado_bloque_modelos)


**Próximo experimento.** Ajustar TF-IDF con el corpus preparado, generar rankings para las mismas 4.208 consultas y compararlos con Recencia. Antes de ejecutarlo cerraremos el desempate y la política para vectores cero. LR supervisada y el ensamble corresponden a pasos posteriores. No buscamos todavía una configuración ganadora entre muchas variantes.

La exploración que sigue conserva los datos y las hipótesis de la iteración anterior. Sirve para interpretar este control y diseñar contenido y actividad; no añade resultados de otros modelos.


## 24. Contenido completo: decisiones, ajuste y evaluación

**Nueva etapa autorizada.** Cerramos desempates y vectores cero, ajustamos TF-IDF solo con el corpus preparado y evaluamos las mismas 4.208 consultas. Después ejecutamos LR y CF + recencia desde entrenamiento reducido, estudiamos actividad y combinamos rankings.

Decisiones fijadas **antes** de mirar estos nuevos resultados:

- Contenido ordena por similitud coseno. Para empates usamos una prioridad pseudoaleatoria fija por oferta, semilla 42, independiente de IDs, fechas y positivos.
- Un candidato con vector cero recibe scores cero para todas las ofertas elegibles y se ordena por esa misma prioridad. Una oferta con vector cero recibe similitud cero. No añadimos Recencia como fallback oculto.
- El vocabulario/IDF queda congelado después del ajuste con perfiles observados en train reducido. Transformar perfiles nuevos no actualiza el vocabulario.
- Se excluyen ofertas futuras antes de seleccionar top-1000 en los métodos propios. Las referencias oficiales conservan su procedimiento; añadiremos un control LR elegible para separar el efecto de esa diferencia.
- MRR es el criterio principal para elegir parámetros, seguido de nDCG@10 en empates. Con métricas idénticas preferimos la configuración más simple, según el orden de enumeración declarado.

Solo exploramos validación. Seleccionar y medir sobre este bloque produce estimaciones optimistas del ganador; los resultados siguientes **no son generalización medida en test**.


In [ ]:
from jth.context import build_context
from jth.ranking import (detail_from_ranks, fast_ranks, lists_from_official, metrics_from_lists, rank_pool,
                         rank_scores, summarize_ranks, verify_official)

ctx = build_context(RAIZ, ruta_train_r, ruta_val_r, ofertas, evaluador_oficial)
assert ctx.gold == gold_recencia
job_ids_modelos, indice_jobs_modelos = ctx.job_ids, ctx.job_index
claves_modelos, gold_indices_modelos, dias_modelos = ctx.keys, ctx.gold_idx, ctx.days
fechas_jobs_modelos, warm_modelos, prioridad_desempate_modelos = ctx.job_dates, ctx.warm, ctx.tie
assert warm_modelos.sum() == n_recencia['warm']

elegibles_modelos = ctx.eligible
def ordenar_scores_modelos(scores, mascara, desempate=None): return rank_scores(ctx, scores, mascara, desempate)
def indices_desde_oficial(preds): return lists_from_official(ctx, preds)
def metricas_listas_modelos(listas): return metrics_from_lists(ctx, listas)
def verificar_oficial_modelos(nombre, listas): return verify_official(ctx, nombre, listas)

resultados_modelos = {}
detalles_modelos = {}
rankings_modelos = {}
rankings_modelos['Recencia'] = indices_desde_oficial(preds_recencia)
resultados_modelos['Recencia'], detalles_modelos['Recencia'] = verificar_oficial_modelos('Recencia', rankings_modelos['Recencia'])


### 24.1. Comprobar desempates y ceros antes del ajuste

El desempate usa una prioridad aleatoria fija y reproducible por oferta. No se elige por validación y no cambia entre configuraciones temporales. El siguiente ejemplo es inventado: los dos primeros scores están empatados, hay un score cero y una oferta ineligible. La máscara debe excluir esa oferta incluso si tiene el score más alto.


In [ ]:
scores_toy_modelos=np.array([.7,.7,0.,9.])
mask_toy_modelos=np.array([True,True,True,False])
tie_toy_modelos=np.array([.8,.2,.5,.1])
idx_toy=np.flatnonzero(mask_toy_modelos)
orden_toy=idx_toy[np.lexsort((tie_toy_modelos[idx_toy],-scores_toy_modelos[idx_toy]))]
assert orden_toy.tolist()==[1,0,2]
ceros_toy=np.zeros(3)
assert np.lexsort((np.array([.8,.2,.5]),-ceros_toy)).tolist()==[1,2,0]
display(pd.DataFrame({'Oferta inventada':['A','B','C'],'Score':[.7,.7,0.],
                      'Prioridad de desempate':[.8,.2,.5],'Orden resultante':[2,1,3]}))
print('Empates, score cero y exclusión de oferta ineligible: OK.')


### 24.2. Ajustar vocabulario e IDF únicamente con entrenamiento

Ahora sí llamamos `fit` con los documentos preparados en 23.8. Luego transformamos los perfiles con el vocabulario congelado. Los perfiles de validación pueden aportar sus atributos para una consulta, pero no contribuyen al IDF. Seguimos asumiendo que la instantánea publicada representa atributos disponibles para el benchmark; no demuestra disponibilidad histórica exacta.


In [ ]:
t_contenido=time.perf_counter()
vectorizador_contenido_preparado.fit(docs_fit_contenido)
vocab_contenido=dict(vectorizador_contenido_preparado.vocabulary_)
idf_contenido=vectorizador_contenido_preparado.idf_.copy()
docs_candidatos_todos=profile_tokens(candidatos)
docs_ofertas_todas=profile_tokens(ofertas)
Q_contenido=vectorizador_contenido_preparado.transform(docs_candidatos_todos)
T_contenido=vectorizador_contenido_preparado.transform(docs_ofertas_todas).T.tocsr()
assert vocab_contenido==vectorizador_contenido_preparado.vocabulary_
assert np.array_equal(idf_contenido,vectorizador_contenido_preparado.idf_)
indice_candidatos_modelos=pd.Index([evaluador_oficial._parse_id(x) for x in candidatos['candidate_id']])
filas_candidatos_consulta=indice_candidatos_modelos.get_indexer([k[0] for k in claves_modelos])
assert (filas_candidatos_consulta>=0).all()
vector_cero_consulta=Q_contenido[filas_candidatos_consulta].getnnz(axis=1)==0
vector_cero_oferta=T_contenido.getnnz(axis=0)==0
display(pd.DataFrame({'Medida':['Documentos de ajuste','Dimensiones aprendidas','Consultas con vector cero',
                              'Ofertas del archivo con vector cero'],
                      'Cantidad':[len(docs_fit_contenido),len(vocab_contenido),int(vector_cero_consulta.sum()),
                                  int(vector_cero_oferta.sum())]}))


### 24.3. Generar los rankings y evaluar

Para cada consulta calculamos coseno, aplicamos la máscara temporal y seleccionamos top-1000 con la prioridad acordada. Reportamos vectores cero y casos sin ninguna similitud positiva como diagnósticos distintos: un vector no nulo también puede no compartir vocabulario con las ofertas elegibles.


In [ ]:
listas_contenido=[]
sin_similitud_positiva=[]
for i,clave in enumerate(claves_modelos):
    scores=(Q_contenido[filas_candidatos_consulta[i]] @ T_contenido).toarray().ravel()
    elegibles=elegibles_modelos(clave[1])
    sin_similitud_positiva.append(not np.any(scores[elegibles]>0))
    listas_contenido.append(ordenar_scores_modelos(scores,elegibles))
rankings_modelos['Contenido']=listas_contenido
resultados_modelos['Contenido'],detalles_modelos['Contenido']=verificar_oficial_modelos('Contenido',listas_contenido)
segundos_contenido=time.perf_counter()-t_contenido
display(pd.DataFrame(resultados_modelos).T.round(5))
diagnosticos_contenido=[]
for nombre,mask in [('Vector cero',vector_cero_consulta),('Vector no cero',~vector_cero_consulta),
                    ('Sin similitud positiva',np.array(sin_similitud_positiva))]:
    diagnosticos_contenido.append({'Grupo':nombre,'Consultas':int(mask.sum()),
        'MRR':detalles_modelos['Contenido']['RR'][mask].mean() if mask.any() else np.nan})
display(pd.DataFrame(diagnosticos_contenido))
fig,axes=plt.subplots(1,2,figsize=(10,3.6))
pd.DataFrame(resultados_modelos).T[['MRR','nDCG@10']].plot.bar(ax=axes[0],rot=0)
pd.DataFrame(resultados_modelos).T[['MRR warm','MRR cold']].plot.bar(ax=axes[1],rot=0)
axes[0].set(title='Temporalidad frente a afinidad',ylabel='Métrica')
axes[1].set(title='Desglose por historial del candidato',ylabel='MRR')
fig.tight_layout(); plt.show()
display(Markdown(f"**Lectura calculada:** contenido cambia el MRR en **{resultados_modelos['Contenido']['MRR']-resultados_modelos['Recencia']['MRR']:+.4f}** frente a Recencia y tardó **{segundos_contenido:.1f} s**. Es un resultado de validación, no una mejora confirmada en test."))


**Análisis.** Esta comparación separa dos señales simples, publicación reciente y afinidad de atributos. Un resultado inferior de contenido no implica que los atributos sean inútiles: puede señalar falta de tiempo, vocabulario imperfecto o diferencias entre perfiles y necesidades. No añadimos una recencia escondida para mejorar el baseline. El siguiente control fuerte será LR, que ya combina atributos y variables temporales.


## 25. Referencias nuevas: LR y CF + recencia desde entrenamiento reducido

Ejecutamos los programas oficiales con los mismos 29.599 procesos de entrenamiento y 4.231 procesos de validación. No reutilizamos las cifras históricas del grupo para esta comparación.

- **LR:** negativo por positivo, semilla 42, modelo logreg, features temporales y LLM, StandardScaler ajustado en train. El programa oficial imputa campos usando los perfiles completos, muestrea negativos de todo el catálogo y conserva peculiaridades de parsing. Son límites heredados de la referencia; no la presentamos como un pipeline de covariables históricas perfectamente reconstruidas.
- **CF + recencia:** ventana oficial de 90 días desde publicación y actualizaciones después de puntuar cada día. No es nuestra futura ventana de actividad. El código usa un catálogo transductivo de IDs de train/val y tiene desempates c2j dependientes del orden de conjuntos; fijamos `PYTHONHASHSEED=42` y no corregimos silenciosamente su algoritmo.
- Ninguna referencia usa etapas finales como etiquetas nuevas ni procesos de validación para entrenar inicialmente. Los atributos sensibles explícitos no forman parte de los feature names usados por LR ni de contenido.

### 19.1. Caché verificable y ejecución oficial

Para evitar repetir una corrida larga en cada `Run All`, se conserva una caché dentro de `data/cache/references/`, que git ignora. Contiene predicciones, logs, manifiestos y scores completos de LR. Solo se reutiliza si coinciden datos, código, versiones, parámetros, particiones y hashes de los artefactos.

El programa `scripts/run_references.py` es visible y reproducible. Llama al `main` oficial de LR e intercepta sus retornos de `predict_proba` para conservar scores antes del top-1000, devolviendo las mismas probabilidades sin modificarlas. Registra además qué matriz llegó a `fit`. CF ejecuta su CLI sin modificar. El `features.csv` incidental de LR queda dentro de la caché, nunca en el repositorio.


In [ ]:
# Las corridas ya ocurrieron en la sección 23 con scripts/run_references.py; aquí comprobamos su identidad.
assert runner_referencias.cache_valid(cache_referencias)
print('Caché e identidad de entrenamiento/validación verificadas:', cache_referencias.relative_to(RAIZ))


### 25.2. Verificar el ajuste y las predicciones

El registro debe mostrar exactamente 29.599 positivos y 29.599 negativos en LR. Alineamos sus scores completos con las consultas y ofertas del notebook. Comprobamos para **todas** las consultas que ordenar esos scores reproduce la lista oficial original antes del filtro de futuras ofertas. Esto permite aplicar actividad antes de truncar, sin fingir que filtrar una lista vieja recupera ofertas que ya estaban fuera.


In [ ]:
fit_lr=json.loads((cache_referencias/'lr/fit.json').read_text())
assert fit_lr['positives']==len(train_reducido) and fit_lr['negatives']==len(train_reducido)
preds_lr=evaluador_oficial.load_preds(cache_referencias/'lr/lr_c2j.pkl.gz')
preds_cf=evaluador_oficial.load_preds(cache_referencias/'cf/cf_c2j.pkl.gz')
keys_cache=[(evaluador_oficial._parse_id(k[0]),k[1]) for k in json.loads((cache_referencias/'keys.json').read_text())]
jobs_cache=[evaluador_oficial._parse_id(k) for k in json.loads((cache_referencias/'jobs.json').read_text())]
assert jobs_cache==job_ids_modelos.tolist()
indices_cache=pd.Index(keys_cache).get_indexer(claves_modelos)
assert (indices_cache>=0).all()
scores_lr_cache=np.load(cache_referencias/'lr/scores_full.npy',mmap_mode='r')
for i,key in enumerate(claves_modelos):
    score=np.asarray(scores_lr_cache[indices_cache[i]])
    reconstruida=job_ids_modelos[np.argsort(-score,kind='stable')[:1000]]
    assert np.array_equal(reconstruida,preds_lr[key])
for nombre,preds in [('LR oficial',preds_lr),('CF + recencia oficial',preds_cf)]:
    rankings_modelos[nombre]=indices_desde_oficial(preds)
    resultados_modelos[nombre],detalles_modelos[nombre]=verificar_oficial_modelos(nombre,rankings_modelos[nombre])
display(pd.DataFrame(resultados_modelos).T.round(5))
display(pd.DataFrame({'Dato de ajuste LR':['Positivos','Negativos','Features no constantes','Iteraciones solver'],
                      'Valor':[fit_lr['positives'],fit_lr['negatives'],len(fit_lr['features']),fit_lr['n_iter']]}))
coef_lr=pd.Series(fit_lr['coefficients'],index=fit_lr['features'])
seleccion_coef=coef_lr.abs().nlargest(10).index
ax=coef_lr.loc[seleccion_coef].sort_values().plot.barh(figsize=(9,4),color='#527aa5')
ax.set(title='Coeficientes LR de mayor magnitud · features estandarizadas',xlabel='Coeficiente',ylabel='')
plt.tight_layout(); plt.show()


**Lectura.** Los coeficientes describen el ajuste de esta referencia condicionada a su muestreo de negativos y su parsing; no son efectos causales ni relevancias comparables entre variables correlacionadas. La salida de LR tampoco es una probabilidad calibrada de contratación: se aprendió a distinguir procesos de negativos muestreados con proporción artificial 1:1. El desglose warm/cold permite comprobar si CF y LR son complementarios en esta validación, sin asumir que la diferencia histórica se repetirá.

Antes de medir actividad añadimos **LR elegible**: mismo score, ofertas futuras excluidas antes del top-1000. Si difiere de LR oficial, ese efecto corresponde al catálogo/corte, no al componente temporal nuevo.


In [ ]:
tie_lr=np.arange(len(ofertas))
listas_lr_elegible=[ordenar_scores_modelos(scores_lr_cache[indices_cache[i]],elegibles_modelos(k[1]),tie_lr)
                    for i,k in enumerate(claves_modelos)]
rankings_modelos['LR elegible']=listas_lr_elegible
resultados_modelos['LR elegible'],detalles_modelos['LR elegible']=verificar_oficial_modelos('LR elegible',listas_lr_elegible)
display(pd.DataFrame(resultados_modelos).T.round(5))
ganancias_cf=detalles_modelos['CF + recencia oficial']['RR']-detalles_modelos['LR oficial']['RR']
display(pd.DataFrame({'Comparación por consulta':['CF mejor que LR','LR mejor que CF','Mismo RR'],
                      'Consultas':[int((ganancias_cf>0).sum()),int((ganancias_cf<0).sum()),int((ganancias_cf==0).sum())]}))
listas_lr_neutral=[ordenar_scores_modelos(np.log(np.clip(scores_lr_cache[indices_cache[i]],1e-15,1)),
                   elegibles_modelos(k[1]),tie_lr) for i,k in enumerate(claves_modelos)]
assert all(np.array_equal(a,b) for a,b in zip(listas_lr_neutral,listas_lr_elegible))
del listas_lr_neutral
print('Penalización suave con λ=0 conserva exactamente LR elegible: OK.')


## 26. Señal propia de actividad, filtro duro y penalización suave

Construimos el estado de cada oferta antes de cada día de validación. Hay dos señales distintas: **popularidad con decaimiento** y **tiempo desde última actividad**. Ambas usan solo procesos anteriores al día; las repeticiones cuentan como procesos, a diferencia de la representación binaria de CF.

La actividad no es una etiqueta de apertura real. Las fechas desconocidas conservan elegibilidad según el evaluador; si tampoco hay historial, la penalización suave será neutral. En estos datos las ofertas tienen fecha de creación, pero dejamos la política especificada.

### 26.1. Estado diario sin información futura

Para popularidad reciente usamos `A(j,t;τ)=Σ exp(-(t-ti)/τ)` con `ti<t`. Para la penalización suave usamos el tiempo desde última actividad; si no existe, usamos edad desde publicación. Un evento del día actual solo modifica el estado para días posteriores.


In [ ]:
from jth.activity import TAUS as TAUS_ACTIVIDAD, daily_states

estados_actividad = daily_states(ctx, train_reducido, validacion, TAUS_ACTIVIDAD)
estados_por_consulta = [estados_actividad[k[1]] for k in claves_modelos]

# Ejemplo inventado de actualización: un evento hoy no estaba disponible esta mañana.
dia_toy_estado=10
ultima_toy=np.array([8.,-np.inf])
assert (dia_toy_estado-ultima_toy)[0]==2 and np.isinf((dia_toy_estado-ultima_toy)[1])
ultima_toy[1]=dia_toy_estado
assert (11-ultima_toy)[1]==1
print('Estado diario preparado:',len(estados_actividad),'días. Actualización posterior y lags previos verificados.')


### 26.2. Baseline propio de popularidad reciente

Esta señal no usa atributos del candidato. Probamos τ de 7, 30 y 90 días, declarados de antemano, para medir cuánto puede explicar la actividad sola. Las ofertas sin actividad tienen score cero, no se eliminan; el desempate usa la prioridad ya fijada. Elegir el mejor τ aquí es ajuste de hiperparámetros sobre validación.


In [ ]:
grid_actividad=[]
mejor_actividad=None
for orden,tau in enumerate(TAUS_ACTIVIDAD):
    listas=[ordenar_scores_modelos(s['decay'][tau],s['eligible']) for s in estados_por_consulta]
    res,det=metricas_listas_modelos(listas)
    grid_actividad.append(dict(res,tau=tau,Orden=orden))
    if mejor_actividad is None or (res['MRR'],res['nDCG@10'])>(mejor_actividad[0]['MRR'],mejor_actividad[0]['nDCG@10']):
        mejor_actividad=(res,listas,det,tau)
grid_actividad=pd.DataFrame(grid_actividad)
rankings_modelos['Actividad propia']=mejor_actividad[1]
resultados_modelos['Actividad propia'],detalles_modelos['Actividad propia']=verificar_oficial_modelos('Actividad propia',mejor_actividad[1])
display(grid_actividad.round(5))


### 26.3. Definir intervenciones y presupuesto de búsqueda

- **Duro:** conservar actividad previa en `w∈{7,14,30,60,90}` días. Comparar sin protección de ofertas nuevas (`g=0`, desactivada) y con gracia por publicación de siete días (`g=7`).
- **Suave:** `log(p_LR)+λ log(g_actividad)`, donde `g_actividad=exp(-lag_efectivo/τ)`. Probamos `τ∈{7,30,90}` y `λ∈{0.25,0.5,1}`. Es un peso heurístico, no probabilidad calibrada de vigencia.
- LR elegible es el control común. Un valor de λ cero debe recuperar ese control exactamente. La multiplicación de probabilidades positivas y la formulación logarítmica son equivalentes salvo el recorte numérico de estabilidad.

El filtro se aplica a **todos los scores LR antes de top-1000**. Si elimina el positivo, la consulta permanece con RR cero. La protección de ofertas nuevas no se basa en el positivo: se calcula para todo el catálogo con su fecha de creación.


In [ ]:
from jth.activity import GRACE_DAYS as GRACIAS_NUEVAS, HARD_WINDOWS as VENTANAS_DURO, SOFT_LAMBDAS as LAMBDAS_SUAVE, SOFT_TAUS as TAUS_SUAVE, hard_mask as mascara_dura_modelos

def cobertura_mascaras_modelos(mascaras):
    return np.array([m[j] for m,j in zip(mascaras,gold_indices_modelos)])

# Prueba inventada: historial reciente, nueva sin historial, vieja sin historial y futura.
estado_toy={'age':np.array([100.,2.,100.,-1.]),'lag':np.array([2.,np.inf,np.inf,np.inf]),
            'eligible':np.array([True,True,True,False])}
assert mascara_dura_modelos(estado_toy,7,0).tolist()==[True,False,False,False]
assert mascara_dura_modelos(estado_toy,7,7).tolist()==[True,True,False,False]
assert np.isclose(np.log(.3)+0*(-200/30),np.log(.3))
print('Pruebas de filtro, rescate de nueva, futuro excluido y peso cero: OK.')

grid_duro=[]; best_duro=None
for w in VENTANAS_DURO:
    for gracia in GRACIAS_NUEVAS:
        mascaras=[mascara_dura_modelos(s,w,gracia) for s in estados_por_consulta]
        listas=[ordenar_scores_modelos(scores_lr_cache[indices_cache[i]],m,tie_lr) for i,m in enumerate(mascaras)]
        res,det=metricas_listas_modelos(listas)
        cobertura=cobertura_mascaras_modelos(mascaras)
        fila=dict(res,w=w,gracia=gracia,Cobertura_pool=100*cobertura.mean(),
                  Positivos_eliminados=int((~cobertura).sum()),Pool_mediano_consulta=float(np.median([m.sum() for m in mascaras])))
        grid_duro.append(fila)
        if best_duro is None or (res['MRR'],res['nDCG@10'])>(best_duro[0]['MRR'],best_duro[0]['nDCG@10']):
            best_duro=(fila,listas,det)
grid_duro=pd.DataFrame(grid_duro)
rankings_modelos['LR + duro']=best_duro[1]
resultados_modelos['LR + duro'],detalles_modelos['LR + duro']=verificar_oficial_modelos('LR + duro',best_duro[1])

grid_suave=[]; best_suave=None
for tau in TAUS_SUAVE:
    for lam in LAMBDAS_SUAVE:
        listas=[]
        for i,s in enumerate(estados_por_consulta):
            score=np.log(np.clip(scores_lr_cache[indices_cache[i]],1e-15,1))-lam*s['effective_lag']/tau
            listas.append(ordenar_scores_modelos(score,s['eligible'],tie_lr))
        res,det=metricas_listas_modelos(listas)
        fila=dict(res,tau=tau,lambda_=lam)
        grid_suave.append(fila)
        if best_suave is None or (res['MRR'],res['nDCG@10'])>(best_suave[0]['MRR'],best_suave[0]['nDCG@10']):
            best_suave=(fila,listas,det)
grid_suave=pd.DataFrame(grid_suave)
rankings_modelos['LR + suave']=best_suave[1]
resultados_modelos['LR + suave'],detalles_modelos['LR + suave']=verificar_oficial_modelos('LR + suave',best_suave[1])
display(grid_duro.round(5)); display(grid_suave.round(5))


**Cómo analizarlo.** La protección de nuevas ofertas puede recuperar cobertura y a la vez añadir competencia al ranking. La ventana que conserva más positivos no necesariamente maximiza MRR. El peso suave conserva el catálogo elegible pero puede desplazar un positivo fuera de top-1000. Presentamos todas las configuraciones y no solo la fila ganadora.

La penalización exponencial depende de `λ/τ`: esos parámetros controlan juntos una única intensidad. Parejas distintas podrían ser equivalentes; no interpretamos sus efectos como independientes. En el resumen congelamos la pareja concreta elegida.


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.8))
grid_duro.pivot(index='w',columns='gracia',values='MRR').rename(columns={0:'Sin gracia',7:'Gracia 7 días'}).plot(ax=axes[0],marker='o')
axes[0].axhline(resultados_modelos['LR elegible']['MRR'],color='gray',ls='--',label='LR elegible')
axes[0].set(title='Filtro duro sobre LR',xlabel='Ventana previa (días)',ylabel='MRR')
grid_duro.pivot(index='w',columns='gracia',values='Cobertura_pool').plot(ax=axes[1],marker='o')
axes[1].set(title='Cobertura del positivo antes del ranking',xlabel='Ventana previa (días)',ylabel='Consultas (%)',ylim=(0,100))
grid_suave.pivot(index='tau',columns='lambda_',values='MRR').plot(ax=axes[2],marker='o')
axes[2].axhline(resultados_modelos['LR elegible']['MRR'],color='gray',ls='--')
axes[2].set(title='Peso suave sobre LR',xlabel='τ (días)',ylabel='MRR')
for ax in axes: ax.grid(alpha=.2)
fig.tight_layout(); plt.show()
display(pd.DataFrame(resultados_modelos).T.round(5))
display(Markdown(f"**Aporte sobre el control comparable LR elegible:** duro **{resultados_modelos['LR + duro']['MRR']-resultados_modelos['LR elegible']['MRR']:+.4f}**, suave **{resultados_modelos['LR + suave']['MRR']-resultados_modelos['LR elegible']['MRR']:+.4f}** de MRR. El mejor filtro duro elimina **{best_duro[0]['Positivos_eliminados']}** positivos antes de ordenar. Son diferencias de validación tras selección de parámetros."))


## 27. Fusionar LR y CF, elegir pesos e incorporar actividad

Primero combinamos las listas oficiales elegibles usando una fusión de rangos ponderada:

`score_RRF(j)=α/(60+r_LR(j)) + (1-α)/(60+r_CF(j))`.

Una oferta ausente de un componente aporta cero en él. La constante 60 queda fija; probamos `α∈{0,0.25,0.5,0.75,1}`. Los extremos deben devolver exactamente la lista del componente correspondiente, sin rellenar con el otro.

También comparamos una compuerta simple: LR para candidatos sin historial y el peso elegido para quienes sí lo tienen. El estado warm se conoce antes de recomendar. Es una variante adicional declarada, no una modificación posterior ocultada al ver resultados.

La fusión trabaja sobre la unión de dos listas truncadas. Es un método válido sobre ese conjunto de recuperación, pero no equivale a fusionar scores CF completos ni garantiza que estén todas las ofertas relevantes. Medimos ese techo de recuperación.


In [ ]:
from jth.fusion import ALPHAS as ALPHAS_FUSION, RRF_C, rrf_pool
listas_lr_base=rankings_modelos['LR oficial']
listas_cf_base=rankings_modelos['CF + recencia oficial']

def pool_fusion_modelos(i,alpha):
    return rrf_pool(listas_lr_base[i],listas_cf_base[i],alpha,len(ofertas),RRF_C)

def ordenar_pool_modelos(pool,scores):
    return rank_pool(ctx,pool,scores)

def rankings_fusion_modelos(alpha,compuerta):
    listas=[]
    for i in range(len(claves_modelos)):
        a=1. if compuerta and not warm_modelos[i] else alpha
        if a==1: listas.append(listas_lr_base[i]); continue
        if a==0: listas.append(listas_cf_base[i]); continue
        pool,s=pool_fusion_modelos(i,a)
        listas.append(ordenar_pool_modelos(pool,s))
    return listas

assert all(np.array_equal(a,b) for a,b in zip(rankings_fusion_modelos(1,False),listas_lr_base))
assert all(np.array_equal(a,b) for a,b in zip(rankings_fusion_modelos(0,False),listas_cf_base))

grid_fusion=[]; best_fusion=None
for compuerta in [False,True]:
    for alpha in ALPHAS_FUSION:
        listas=rankings_fusion_modelos(alpha,compuerta)
        res,det=metricas_listas_modelos(listas)
        fila=dict(res,alpha_LR=alpha,compuerta=compuerta)
        grid_fusion.append(fila)
        if best_fusion is None or (res['MRR'],res['nDCG@10'])>(best_fusion[0]['MRR'],best_fusion[0]['nDCG@10']):
            best_fusion=(fila,listas,det)
grid_fusion=pd.DataFrame(grid_fusion)
rankings_modelos['Fusión LR-CF']=best_fusion[1]
resultados_modelos['Fusión LR-CF'],detalles_modelos['Fusión LR-CF']=verificar_oficial_modelos('Fusión LR-CF',best_fusion[1])
display(grid_fusion.round(5))
ax=grid_fusion.pivot(index='alpha_LR',columns='compuerta',values='MRR').rename(columns={False:'Peso global',True:'LR en cold'}).plot(marker='o',figsize=(8,3.4))
ax.set(title='Selección del peso de fusión en validación',xlabel='α: peso de LR',ylabel='MRR')
ax.grid(alpha=.2); plt.tight_layout(); plt.show()

pool_ganador=[]; scores_ganador=[]
for i in range(len(claves_modelos)):
    a=1. if best_fusion[0]['compuerta'] and not warm_modelos[i] else best_fusion[0]['alpha_LR']
    pool,s=pool_fusion_modelos(i,a)
    pool_ganador.append(pool); scores_ganador.append(s)
cobertura_pool_fusion=np.array([g in p for g,p in zip(gold_indices_modelos,pool_ganador)])
print('Positivos presentes en el pool de fusión elegido:',int(cobertura_pool_fusion.sum()),'/',len(claves_modelos))


### 27.1. Añadir actividad a la fusión seleccionada

Fijamos primero su α y su compuerta. Después probamos el mismo presupuesto de diez cortes duros y nueve pesos suaves sobre su pool. La variante suave ordena por `log(score_RRF) - λ·lag_efectivo/τ`. No modifica las listas base ni usa atributos futuros.

Aquí el filtro se aplica antes del top-1000 **de la fusión**, pero dentro de su pool de recuperación. No lo confundimos con el filtro sobre el catálogo completo de LR. Reportamos pérdidas adicionales y conservamos todas las consultas, incluidas las que quedan con lista vacía.


In [ ]:
grid_fusion_temporal=[]; best_fusion_temporal=None
config_temporales=[('duro',w,g,None,None) for w in VENTANAS_DURO for g in GRACIAS_NUEVAS]
config_temporales += [('suave',None,None,tau,lam) for tau in TAUS_SUAVE for lam in LAMBDAS_SUAVE]
for tipo,w,gracia,tau,lam in config_temporales:
    listas=[]; conservados=[]
    for i,(pool,s) in enumerate(zip(pool_ganador,scores_ganador)):
        estado=estados_por_consulta[i]
        if tipo=='duro':
            mantener=mascara_dura_modelos(estado,w,gracia)[pool]
            p,ss=pool[mantener],s[mantener]
        else:
            p=pool
            ss=np.log(np.maximum(s,1e-15))-lam*estado['effective_lag'][p]/tau
        conservados.append(gold_indices_modelos[i] in p)
        listas.append(ordenar_pool_modelos(p,ss))
    res,det=metricas_listas_modelos(listas)
    fila=dict(res,tipo=tipo,w=w,gracia=gracia,tau=tau,lambda_=lam,
              Cobertura_pool=100*np.mean(conservados),Positivos_fuera_pool=int(len(conservados)-np.sum(conservados)))
    grid_fusion_temporal.append(fila)
    if best_fusion_temporal is None or (res['MRR'],res['nDCG@10'])>(best_fusion_temporal[0]['MRR'],best_fusion_temporal[0]['nDCG@10']):
        best_fusion_temporal=(fila,listas,det)
grid_fusion_temporal=pd.DataFrame(grid_fusion_temporal)
rankings_modelos['Fusión + actividad']=best_fusion_temporal[1]
resultados_modelos['Fusión + actividad'],detalles_modelos['Fusión + actividad']=verificar_oficial_modelos('Fusión + actividad',best_fusion_temporal[1])
display(grid_fusion_temporal.round(5))
display(Markdown(f"**Incremento de actividad sobre la fusión seleccionada:** **{resultados_modelos['Fusión + actividad']['MRR']-resultados_modelos['Fusión LR-CF']['MRR']:+.4f}** de MRR. Un valor negativo debe conservarse como resultado; la complejidad extra no se adopta automáticamente."))


## 33. Comparación final de esta etapa, incertidumbre y configuración elegida

Mostramos referencias y mejores variantes por familia. El mejor resultado explorado puede ser una referencia o una variante sin CF; no forzamos que gane el sistema más complejo. Los valores de validación del ganador están condicionados a la búsqueda realizada y deben contrastarse en una evaluación posterior con decisiones congeladas.

Cuantificamos además diferencias pareadas frente a LR oficial mediante bootstrap de bloques consecutivos de siete días, 2.000 remuestreos y semilla 42. Se remuestrean los mismos bloques en ambos modelos. Es una aproximación a la dependencia temporal; no elimina dependencias de candidatos entre bloques ni corrige el optimismo de selección de hiperparámetros.


In [ ]:
tabla_final_modelos=pd.DataFrame(resultados_modelos).T.sort_values(['MRR','nDCG@10'],ascending=False)
display(tabla_final_modelos.round(6))
comparaciones_incrementales=[('Contenido','Recencia'),('LR elegible','LR oficial'),
    ('LR + duro','LR elegible'),('LR + suave','LR elegible'),
    ('Fusión LR-CF','LR oficial'),('Fusión + actividad','Fusión LR-CF')]
tabla_aportes=pd.DataFrame([
    {'Variante':v,'Control':c,'ΔMRR':resultados_modelos[v]['MRR']-resultados_modelos[c]['MRR'],
     'ΔMRR warm':resultados_modelos[v]['MRR warm']-resultados_modelos[c]['MRR warm'],
     'ΔMRR cold':resultados_modelos[v]['MRR cold']-resultados_modelos[c]['MRR cold']}
    for v,c in comparaciones_incrementales])
display(tabla_aportes.round(6))
ganador_modelos=tabla_final_modelos.index[0]
from jth.uncertainty import BlockBootstrap
bootstrap=BlockBootstrap(dias_modelos)
filas_ic=[]
for nombre,detalle in detalles_modelos.items():
    media,low,high=bootstrap.difference(detalle['RR'],detalles_modelos['LR oficial']['RR'])
    filas_ic.append({'Modelo':nombre,'ΔMRR vs LR oficial':media,'IC95 inferior':low,'IC95 superior':high})
intervalos_modelos=pd.DataFrame(filas_ic).set_index('Modelo')
display(intervalos_modelos.round(6))
print('Consultas:',len(claves_modelos),'| Bloques temporales:',len(bootstrap.unique),'| Remuestreos:',len(bootstrap.resamples))
pares_incrementales=comparaciones_incrementales+[('Fusión + actividad','LR + suave')]
ic_incrementales=[]
for variante,control in pares_incrementales:
    media,low,high=bootstrap.difference(detalles_modelos[variante]['RR'],detalles_modelos[control]['RR'])
    ic_incrementales.append({'Variante':variante,'Control':control,'ΔMRR':media,
                            'IC95 inferior':low,'IC95 superior':high})
display(pd.DataFrame(ic_incrementales).round(6))

fig,axes=plt.subplots(1,2,figsize=(13,4.5))
tabla_final_modelos[['MRR','nDCG@10']].plot.barh(ax=axes[0])
tabla_final_modelos[['MRR warm','MRR cold']].plot.barh(ax=axes[1])
axes[0].set(title='Referencias y mejores variantes · validación',xlabel='Métrica',ylabel='')
axes[1].set(title='Aporte por historial del candidato',xlabel='MRR',ylabel='')
fig.tight_layout(); plt.show()

configuracion_seleccionada={
    'criterio':'MRR; nDCG@10 secundario; primera configuración enumerada en empate exacto',
    'ganador_validacion':ganador_modelos,
    'actividad_tau':mejor_actividad[3],
    'lr_duro':{'w':int(best_duro[0]['w']),'gracia':int(best_duro[0]['gracia'])},
    'lr_suave':{'tau':int(best_suave[0]['tau']),'lambda':float(best_suave[0]['lambda_'])},
    'fusion':{'alpha_LR':float(best_fusion[0]['alpha_LR']),'compuerta':bool(best_fusion[0]['compuerta']),'RRF_c':RRF_C},
    'fusion_actividad':{k:best_fusion_temporal[0][k] for k in ['tipo','w','gracia','tau','lambda_']},
    'configuraciones_exploradas':int(len(grid_actividad)+len(grid_duro)+len(grid_suave)+len(grid_fusion)+len(grid_fusion_temporal)),
    'cache_referencias':str(cache_referencias.relative_to(RAIZ)),
    'evaluacion_test':'No realizada en esta etapa; no se utilizó para seleccionar parámetros.',
}
display(Markdown('**Configuración y presupuesto registrados en el notebook:**'))
print(json.dumps(configuracion_seleccionada,indent=2,ensure_ascii=False))
display(Markdown(f"**Resultado de esta etapa:** la mejor familia explorada es **{ganador_modelos}**, con MRR **{tabla_final_modelos.iloc[0]['MRR']:.4f}**. La diferencia respecto de LR oficial es **{tabla_final_modelos.iloc[0]['MRR']-resultados_modelos['LR oficial']['MRR']:+.4f}**. La tabla de intervalos y la exposición a validación deben acompañar esa cifra; no es todavía rendimiento final en test."))
display(Markdown(
    f"**Comparación de los dos finalistas:** fusión + actividad cambia el MRR en "
    f"**{resultados_modelos['Fusión + actividad']['MRR']-resultados_modelos['LR + suave']['MRR']:+.4f}** "
    f"y el acierto top-10 en **{100*(resultados_modelos['Fusión + actividad']['HR@10']-resultados_modelos['LR + suave']['HR@10']):+.2f} puntos porcentuales** "
    "respecto de LR + suave. Es un intercambio entre métricas y complejidad. Las dos variantes tienen puntuaciones y parámetros temporales distintos; esta comparación no aísla exclusivamente el aporte de CF."
))


**Qué decisiones quedan sustentadas.** Podemos distinguir contenido puro, actividad pura, LR/CF, efecto de elegibilidad, efecto de filtrar o penalizar y efecto de fusionar. Las tablas de búsqueda completas permiten revisar los parámetros. Si la mezcla o actividad no mejoran, conservaremos la referencia más simple; una ganancia seleccionada aquí sigue necesitando evaluación posterior.

**Límites de esta iteración.** Los perfiles son una instantánea; LR y CF conservan peculiaridades oficiales documentadas; la búsqueda secuencial no explora todas las interacciones posibles y puede sobreajustar validación. Los IC son condicionales al procedimiento y no deben presentarse como prueba concluyente de generalización. La etiqueta sigue siendo proceso registrado, no contratación, y ausencia de actividad no equivale a cierre.

**Siguiente etapa del roadmap:** decidir qué configuración congelar, hacer una comprobación de robustez temporal si corresponde y avanzar al perfil dinámico de ofertas mediante promedio de candidatos anteriores. El modelo secuencial sigue condicionado al promedio simple. La exploración profunda previa se conserva a continuación.


**Lectura de los finalistas en la ejecución guardada.** Fusión + actividad gana por el criterio principal MRR, pero su diferencia frente a LR + suave es de aproximadamente 0,0082. El IC pareado por bloques para esa diferencia va de aproximadamente −0,0002 a 0,0180 e incluye cero. Esta validación no permite afirmar una superioridad firme de la alternativa más compleja sobre LR + suave. Además, LR + suave alcanza más aciertos top-10 y mejor MRR cold, mientras la fusión con actividad mejora el grupo warm. Conservamos ambos como candidatos, con la configuración de mayor MRR identificada para contrastarla posteriormente. Estos intervalos son condicionales y no corrigen la búsqueda de hiperparámetros.


## 38. Síntesis: convertir la exploración en decisiones de implementación

La tabla siguiente reúne evidencia calculada en esta ejecución. Para volver a comprobar una cifra, ejecuta su sección anterior. Los resultados de composición global, entrenamiento y validación están identificados; no son todos la misma población.


In [ ]:
fila30 = resumen_ventanas.loc[30]
hallazgos_eda = pd.DataFrame([
    {'Pregunta': '¿Qué tan escaso es el historial de candidatos?',
     'Evidencia': f'{100 * grados_candidatos.eq(1).mean():.1f}% con un solo proceso en train reducido',
     'Implicación': 'Mantener contenido y evaluar candidatos nuevos separadamente.'},
    {'Pregunta': '¿Se repite un par candidato-oferta?',
     'Evidencia': f'{historial.duplicated(["candidate_id", "job_id"]).sum():,} filas adicionales en pares repetidos del registro',
     'Implicación': 'Conservar procesos; documentar si un modelo binariza pares.'},
    {'Pregunta': '¿La experiencia siempre es numérica?',
     'Evidencia': 'La tabla de formatos distingue rangos de texto y números.',
     'Implicación': 'Codificar según semántica; no convertir rangos a NaN ni booleanos.'},
    {'Pregunta': '¿Hay contenido sin historial del candidato?',
     'Evidencia': f'{len(vocab_c & vocab_j):,} tokens compartidos entre perfiles de entrenamiento',
     'Implicación': 'Preparar contenido con vocabulario/IDF aprendido solo en train.'},
    {'Pregunta': '¿Cuántos candidatos llegan sin historial?',
     'Evidencia': f'{100 * (~consultas_validacion.candidato_con_historial).mean():.1f}% de consultas de validación',
     'Implicación': 'Comprobar fallback de CF y posible compuerta por historial.'},
    {'Pregunta': '¿Las ofertas acumulan historial utilizable?',
     'Evidencia': f'{100 * consultas_validacion.oferta_con_historial.mean():.1f}% de consultas de validación con oferta previamente observada',
     'Implicación': 'Evaluar el promedio temporal de candidatos anteriores de cada oferta.'},
    {'Pregunta': '¿Qué conserva una ventana de 30 días?',
     'Evidencia': f'{fila30["Cobertura por consulta (%)"]:.1f}% de positivos; {fila30["ofertas_pool_mediana"]:.0f} ofertas por día (mediana), en validación',
     'Implicación': 'Es diagnóstico de cobertura; falta evaluar ranking, pérdidas y costo.'},
])
display(hallazgos_eda)


### Próximo paso de la bitácora

1. Control de Recencia completado en la sección 23, con evaluador oficial y métricas por consulta.
2. Contenido, referencias nuevas, actividad y fusión ejecutados en las secciones 24–27; revisar sus resultados y límites antes de elegir el sistema final.
3. Las búsquedas de actividad conservan todas las consultas y se registran completas; falta la evaluación final con decisiones congeladas.
4. El próximo modelo nuevo es el perfil dinámico de ofertas, con un promedio simple antes de cualquier secuencia.

La exploración de estas secciones aporta contexto; las secciones 24–27 ya ajustan modelos y comparan resultados en validación. Las mejoras seleccionadas allí todavía requieren evaluación final con decisiones congeladas. Las fechas perturbadas, los perfiles como instantánea y los procesos curados por reclutadores limitan lo que podemos inferir sobre disponibilidad y preferencias reales.

### Forma de trabajo

Las tablas, verificaciones y figuras quedan en este notebook; no se exportan `.tex` ni imágenes externas. Las referencias usan CSV, predicciones, scores y logs en la caché local ignorada por git descrita en 25.1. El divisor y los ejemplos sintéticos usan carpetas temporales. Las aserciones verifican consistencia, no calidad predictiva. No se añadieron dependencias ni se modificaron los algoritmos o el protocolo principal.

Las indicaciones del repositorio sobre tablas `.tex` se refieren a alimentar el informe en LaTeX. Si necesitamos una exportación para una entrega, añadiremos una celda explícita con su propósito.
